# Global Automotive Investment Database — Block 10

## Global Fundamentals, Point-in-Time Research Panel and Model Exports

This notebook combines the regional fundamentals from Blocks 3–8 with the accepted quality-control decisions from Block 9.

It publishes:

1. a harmonised global fundamental fact store;
2. point-in-time monthly accounting panels;
3. model-ready issuer-month features for latent automotive factor detection;
4. QuantConnect-compatible long-format fundamentals and security mappings;
5. complete source, quality and decision lineage.

Block 9 may improve accepted mappings and classifications, but it never overwrites a deterministic reported value. Missing and stale observations remain missing rather than being converted to zero.

In [ ]:
# 1. INSTALLS AND IMPORTS

!pip -q install pandas numpy pyarrow tqdm psutil

from __future__ import annotations

import ctypes
import gc
import hashlib
import json
import re
import types

from datetime import date, datetime, timezone
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil

from tqdm.auto import tqdm

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("xlsxwriter") is None:
    print(
        "Installing xlsxwriter..."
    )

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "xlsxwriter",
    ])

    print(
        "xlsxwriter installed."
    )

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 240)

print("Python environment ready.")


Installing xlsxwriter...
xlsxwriter installed.
Python environment ready.


In [ ]:
# 2. SETTINGS, DIRECTORIES AND REGIONAL CONTRACTS

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy"
    )
else:
    PROJECT_ROOT = Path(
        "/content/global_automotive_investment_database"
    )

DATA_ROOT = PROJECT_ROOT / "data"
INTERIM_ROOT = DATA_ROOT / "interim"

BLOCK_10_OUTPUT_DIR = INTERIM_ROOT / "block_10"
BLOCK_10_MANIFEST_PATH = (
    BLOCK_10_OUTPUT_DIR / "block_10_manifest.json"
)


BLOCK_9_MANIFEST_PATH = (
    INTERIM_ROOT / "block_9" / "block_9_manifest.json"
)

BLOCK_9_ACCEPTED_TABLE_ALIASES = [
    "ai_exception_deduplicated_outcomes_df",
    "all_accepted_signature_decisions_df",
    "accepted_ai_enrichments_df",
]

BLOCK_10_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REGIONAL_MANIFESTS = {
    "USA": INTERIM_ROOT / "block_3" / "block_3_manifest.json",
    "EUROPE": INTERIM_ROOT / "block_4" / "block_4_manifest.json",
    "JAPAN": INTERIM_ROOT / "block_5" / "block_5_manifest.json",
    "KOREA": INTERIM_ROOT / "block_6" / "block_6_manifest.json",
    "HONG_KONG": INTERIM_ROOT / "block_7" / "block_7_manifest.json",
    "MAINLAND_CHINA": INTERIM_ROOT / "block_7" / "block_7_manifest.json",
    "AUSTRALIA": INTERIM_ROOT / "block_8" / "block_8_manifest.json",
}

# Aliases allow Block 10 to load both current and earlier regional contracts.
REGIONAL_TABLE_ALIASES = {
    "USA": {
        "standardised_facts": [
            "sec_fundamentals_standardised_df",
            "usa_fundamentals_standardised_df",
        ],
        "incremental_facts": [
            "sec_incremental_facts_df",
            "usa_incremental_facts_df",
        ],
        "filings": [
            "sec_filing_metadata_df",
            "usa_filing_metadata_df",
        ],
        "concept_dictionary": [
            "sec_standard_concept_dictionary_df",
            "usa_standard_concept_dictionary_df",
        ],
        "issuer_universe": [
            "usa_economic_issuer_universe_df",
            "usa_issuer_universe_df",
        ],
        "security_universe": [
            "usa_issuer_security_universe_df",
            "usa_security_universe_df",
        ],
        "relationship_graph": [
            "usa_entity_relationship_graph_df",
        ],
    },
    "EUROPE": {
        "standardised_facts": [
            "europe_fundamentals_standardised_df",
        ],
        "incremental_facts": [
            "europe_incremental_facts_df",
        ],
        "filings": [
            "europe_filing_metadata_df",
        ],
        "concept_dictionary": [
            "europe_standard_concept_dictionary_df",
        ],
        "issuer_universe": [
            "europe_economic_issuer_universe_df",
            "europe_issuer_universe_df",
        ],
        "security_universe": [
            "europe_issuer_security_universe_df",
            "europe_security_universe_df",
        ],
        "relationship_graph": [
            "europe_entity_relationship_graph_df",
        ],
    },
    "JAPAN": {
        "standardised_facts": [
            "japan_fundamentals_standardised_df",
        ],
        "incremental_facts": [
            "japan_incremental_facts_df",
        ],
        "filings": [
            "japan_filing_metadata_df",
        ],
        "concept_dictionary": [
            "japan_standard_concept_dictionary_df",
        ],
        "issuer_universe": [
            "japan_economic_issuer_universe_df",
            "japan_issuer_universe_df",
        ],
        "security_universe": [
            "japan_issuer_security_universe_df",
            "japan_security_universe_df",
        ],
        "relationship_graph": [
            "japan_entity_relationship_graph_df",
        ],
    },
    "KOREA": {
        "standardised_facts": [
            "korea_fundamentals_standardised_df",
        ],
        "incremental_facts": [
            "korea_incremental_facts_df",
        ],
        "filings": [
            "korea_filing_metadata_df",
        ],
        "concept_dictionary": [
            "korea_standard_concept_dictionary_df",
        ],
        "issuer_universe": [
            "korea_economic_issuer_universe_df",
            "korea_issuer_universe_df",
        ],
        "security_universe": [
            "korea_issuer_security_universe_df",
            "korea_security_universe_df",
        ],
        "relationship_graph": [
            "korea_entity_relationship_graph_df",
        ],
    },
    "HONG_KONG": {
        "standardised_facts": [
            "hong_kong_fundamentals_standardised_df",
            "hk_fundamentals_standardised_df",
        ],
        "incremental_facts": [
            "hong_kong_incremental_facts_df",
            "hk_incremental_facts_df",
        ],
        "filings": [
            "hong_kong_filing_metadata_df",
            "hk_disclosure_inventory_df",
        ],
        "concept_dictionary": [
            "hong_kong_standard_concept_dictionary_df",
            "hk_standard_concept_dictionary_df",
        ],
        "issuer_universe": [
            "hong_kong_economic_issuer_universe_df",
            "hk_economic_issuer_universe_df",
        ],
        "security_universe": [
            "hong_kong_issuer_security_universe_df",
            "hk_issuer_security_universe_df",
        ],
        "relationship_graph": [
            "hong_kong_entity_relationship_graph_df",
            "hk_entity_relationship_graph_df",
        ],
    },
    "MAINLAND_CHINA": {
        "standardised_facts": [
            "china_fundamentals_standardised_df",
            "mainland_china_fundamentals_standardised_df",
        ],
        "incremental_facts": [
            "china_incremental_facts_df",
            "mainland_china_incremental_facts_df",
        ],
        "filings": [
            "china_disclosure_inventory_df",
            "mainland_china_filing_metadata_df",
        ],
        "concept_dictionary": [
            "china_standard_concept_dictionary_df",
            "mainland_china_standard_concept_dictionary_df",
        ],
        "issuer_universe": [
            "china_economic_issuer_universe_df",
            "mainland_china_economic_issuer_universe_df",
        ],
        "security_universe": [
            "china_issuer_security_universe_df",
            "mainland_china_issuer_security_universe_df",
        ],
        "relationship_graph": [
            "china_entity_relationship_graph_df",
            "mainland_china_entity_relationship_graph_df",
        ],
    },
    "AUSTRALIA": {
        "standardised_facts": [
            "australia_fundamentals_standardised_df",
        ],
        "incremental_facts": [
            "australia_incremental_facts_df",
        ],
        "filings": [
            "australia_filing_metadata_df",
        ],
        "concept_dictionary": [
            "australia_standard_concept_dictionary_df",
        ],
        "issuer_universe": [
            "australia_issuer_universe_df",
            "australia_economic_issuer_universe_df",
        ],
        "security_universe": [
            "australia_security_universe_df",
            "australia_issuer_security_universe_df",
        ],
        "relationship_graph": [
            "australia_entity_relationship_graph_df",
        ],
    },
}

PERSIST_BLOCK_10_OUTPUTS = True
OVERWRITE_PERSISTED_OUTPUTS = True
PARQUET_CHUNK_ROWS = 10_000
REQUIRE_ALL_REGIONS = False
RETAIN_REGIONAL_EXTRA_COLUMNS = False

# Monthly interpretation panel.
MONTH_END_FREQUENCY = "ME"
PANEL_START_DATE = pd.Timestamp("2019-12-31")
PANEL_END_DATE = pd.Timestamp.today().normalize() + pd.offsets.MonthEnd(0)

# Conservative maximum age for monthly carry-forward.
DEFAULT_MAX_STALENESS_DAYS = 550
FLOW_MAX_STALENESS_DAYS = 550
INSTANT_MAX_STALENESS_DAYS = 550

CORE_INTERPRETATION_CONCEPTS = [
    "revenue",
    "gross_profit",
    "operating_income",
    "ebitda",
    "net_income",
    "cash_and_cash_equivalents",
    "inventory",
    "total_assets",
    "total_liabilities",
    "total_equity",
    "total_debt",
    "operating_cash_flow",
    "capital_expenditure",
    "research_development",
    "intangible_assets",
]

print("Project root:", PROJECT_ROOT)
print("Block 10 output directory:", BLOCK_10_OUTPUT_DIR)


Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy
Block 10 output directory: /content/drive/MyDrive/Colab Notebooks/00 A1 Auto Factor Strategy/data/interim/block_10


In [ ]:
# 3. MANIFEST AND TABLE-LOADING HELPERS

def read_manifest(manifest_path: Path) -> dict:
    if not manifest_path.exists():
        raise FileNotFoundError(manifest_path)

    with manifest_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def manifest_table_index(manifest: dict) -> dict:
    return {
        item["table_name"]: item
        for item in manifest.get(
            "tables",
            [],
        )
    }


def resolve_manifest_table_path(
    manifest_path: Path,
    table_name: str,
) -> Path:
    manifest = read_manifest(
        manifest_path
    )
    index = manifest_table_index(
        manifest
    )

    if table_name not in index:
        raise KeyError(
            f"{table_name} is not present in "
            f"{manifest_path.name}"
        )

    path_value = index[
        table_name
    ].get("path")

    if path_value:
        path = Path(path_value)
    else:
        path = (
            manifest_path.parent
            / f"{table_name}.parquet"
        )

    if not path.is_absolute():
        path = (
            manifest_path.parent
            / path
        ).resolve()

    return path


def load_manifest_table(
    manifest_path: Path,
    table_name: str,
    columns=None,
) -> pd.DataFrame:
    path = resolve_manifest_table_path(
        manifest_path,
        table_name,
    )

    if not path.exists():
        raise FileNotFoundError(path)

    return pd.read_parquet(
        path,
        columns=columns,
    )


def first_available_alias(
    manifest_path: Path,
    aliases: list[str],
):
    if not manifest_path.exists():
        return None

    manifest = read_manifest(
        manifest_path
    )
    available = set(
        manifest_table_index(
            manifest
        )
    )

    return next(
        (
            alias
            for alias in aliases
            if alias in available
        ),
        None,
    )


def load_optional_alias(
    region: str,
    table_role: str,
):
    manifest_path = REGIONAL_MANIFESTS[
        region
    ]
    aliases = REGIONAL_TABLE_ALIASES[
        region
    ].get(
        table_role,
        [],
    )

    if not manifest_path.exists():
        return (
            pd.DataFrame(),
            None,
            "MANIFEST_MISSING",
            pd.NA,
        )

    table_name = first_available_alias(
        manifest_path,
        aliases,
    )

    if table_name is None:
        return (
            pd.DataFrame(),
            None,
            "TABLE_NOT_FOUND",
            pd.NA,
        )

    try:
        frame = load_manifest_table(
            manifest_path,
            table_name,
        )

        return (
            frame,
            table_name,
            "LOADED",
            pd.NA,
        )

    except Exception as exc:
        return (
            pd.DataFrame(),
            table_name,
            "FAILED",
            repr(exc),
        )


def coalesce_columns(
    dataframe,
    candidates,
    default=pd.NA,
):
    available = [
        column
        for column in candidates
        if column in dataframe.columns
    ]

    if not available:
        return pd.Series(
            default,
            index=dataframe.index,
            dtype="object",
        )

    result = dataframe[
        available[0]
    ].copy()

    for column in available[1:]:
        result = result.combine_first(
            dataframe[column]
        )

    return result


regional_manifest_status_rows = []

for region, manifest_path in (
    REGIONAL_MANIFESTS.items()
):
    regional_manifest_status_rows.append({
        "source_region": region,
        "manifest_path": str(
            manifest_path
        ),
        "manifest_exists": (
            manifest_path.exists()
        ),
    })

regional_manifest_status_df = pd.DataFrame(
    regional_manifest_status_rows
)

if (
    REQUIRE_ALL_REGIONS
    and not regional_manifest_status_df[
        "manifest_exists"
    ].all()
):
    missing = regional_manifest_status_df[
        ~regional_manifest_status_df[
            "manifest_exists"
        ]
    ]["source_region"].tolist()

    raise RuntimeError(
        f"Missing required regional manifests: {missing}"
    )

display(
    regional_manifest_status_df
)


,source_region,manifest_path,manifest_exists
0,USA,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,True
1,EUROPE,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,True
2,JAPAN,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,True
3,KOREA,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,True
4,HONG_KONG,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,True
5,MAINLAND_CHINA,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,True
6,AUSTRALIA,/content/drive/MyDrive/Colab Notebooks/00 A1 A...,True


In [ ]:
# 4. LOAD REGIONAL ISSUER, SECURITY AND RELATIONSHIP ARCHITECTURE

issuer_frames = []
security_frames = []
relationship_frames = []
architecture_load_rows = []


def harmonise_issuer_universe(
    region,
    dataframe,
    source_table,
):
    output = pd.DataFrame(
        index=dataframe.index
    )

    output["issuer_id"] = coalesce_columns(
        dataframe,
        [
            "issuer_id",
            "economic_issuer_id",
            "global_issuer_id",
        ],
    )

    output["issuer_name"] = coalesce_columns(
        dataframe,
        [
            "issuer_name",
            "canonical_issuer_name",
            "company_name",
            "entity_name",
            "name",
        ],
    )

    output["country_of_incorporation"] = (
        coalesce_columns(
            dataframe,
            [
                "country_of_incorporation",
                "incorporation_country",
                "country",
            ],
        )
    )

    output["reporting_currency"] = (
        coalesce_columns(
            dataframe,
            [
                "reporting_currency",
                "currency",
                "functional_currency",
            ],
        )
    )

    output["source_region"] = region
    output["source_table"] = (
        source_table
    )

    return output


def harmonise_security_universe(
    region,
    dataframe,
    source_table,
):
    output = pd.DataFrame(
        index=dataframe.index
    )

    output["security_id"] = coalesce_columns(
        dataframe,
        [
            "security_id",
            "global_security_id",
        ],
    )

    output["issuer_id"] = coalesce_columns(
        dataframe,
        [
            "issuer_id",
            "economic_issuer_id",
            "global_issuer_id",
        ],
    )

    output["issuer_name"] = coalesce_columns(
        dataframe,
        [
            "issuer_name",
            "canonical_issuer_name",
            "company_name",
            "entity_name",
        ],
    )

    output["ticker"] = coalesce_columns(
        dataframe,
        [
            "ticker",
            "asx_code",
            "symbol",
            "local_symbol",
        ],
    )

    output["exchange"] = coalesce_columns(
        dataframe,
        [
            "exchange",
            "exchange_code",
            "mic",
        ],
    )

    output["isin"] = coalesce_columns(
        dataframe,
        ["isin", "ISIN"],
    )

    output["security_type"] = coalesce_columns(
        dataframe,
        [
            "security_type",
            "instrument_type",
            "share_class",
        ],
    )

    output["is_adr"] = coalesce_columns(
        dataframe,
        [
            "is_adr",
            "adr_flag",
            "is_depositary_receipt",
        ],
        default=False,
    )

    output["source_region"] = region
    output["source_table"] = (
        source_table
    )

    return output


def harmonise_relationship_graph(
    region,
    dataframe,
    source_table,
):
    output = pd.DataFrame(
        index=dataframe.index
    )

    output["from_entity_id"] = (
        coalesce_columns(
            dataframe,
            [
                "from_entity_id",
                "source_entity_id",
                "parent_issuer_id",
                "issuer_id",
            ],
        )
    )

    output["to_entity_id"] = coalesce_columns(
        dataframe,
        [
            "to_entity_id",
            "target_entity_id",
            "child_issuer_id",
            "related_issuer_id",
        ],
    )

    output["relationship_type"] = (
        coalesce_columns(
            dataframe,
            [
                "relationship_type",
                "relation_type",
                "edge_type",
            ],
        )
    )

    output["effective_start"] = (
        pd.to_datetime(
            coalesce_columns(
                dataframe,
                [
                    "effective_start",
                    "start_date",
                    "valid_from",
                ],
            ),
            errors="coerce",
        )
    )

    output["effective_end"] = (
        pd.to_datetime(
            coalesce_columns(
                dataframe,
                [
                    "effective_end",
                    "end_date",
                    "valid_to",
                ],
            ),
            errors="coerce",
        )
    )

    output["confidence"] = (
        pd.to_numeric(
            coalesce_columns(
                dataframe,
                [
                    "confidence",
                    "relationship_confidence",
                ],
            ),
            errors="coerce",
        )
    )

    output["source_region"] = region
    output["source_table"] = (
        source_table
    )

    return output


for region in REGIONAL_MANIFESTS:
    for role in [
        "issuer_universe",
        "security_universe",
        "relationship_graph",
    ]:
        (
            frame,
            table_name,
            status,
            error,
        ) = load_optional_alias(
            region,
            role,
        )

        architecture_load_rows.append({
            "source_region": region,
            "table_role": role,
            "table_name": table_name,
            "status": status,
            "row_count": len(frame),
            "error": error,
        })

        if status != "LOADED":
            continue

        if role == "issuer_universe":
            issuer_frames.append(
                harmonise_issuer_universe(
                    region,
                    frame,
                    table_name,
                )
            )

        elif role == "security_universe":
            security_frames.append(
                harmonise_security_universe(
                    region,
                    frame,
                    table_name,
                )
            )

        else:
            relationship_frames.append(
                harmonise_relationship_graph(
                    region,
                    frame,
                    table_name,
                )
            )


global_economic_issuer_universe_df = (
    pd.concat(
        issuer_frames,
        ignore_index=True,
        sort=False,
    )
    if issuer_frames
    else pd.DataFrame(
        columns=[
            "issuer_id",
            "issuer_name",
            "country_of_incorporation",
            "reporting_currency",
            "source_region",
            "source_table",
        ]
    )
)

global_economic_issuer_universe_df = (
    global_economic_issuer_universe_df
    .dropna(
        subset=[
            "issuer_id",
        ]
    )
    .sort_values(
        [
            "issuer_id",
            "issuer_name",
        ],
        na_position="last",
    )
    .drop_duplicates(
        "issuer_id",
        keep="first",
    )
    .reset_index(drop=True)
)


global_issuer_security_universe_df = (
    pd.concat(
        security_frames,
        ignore_index=True,
        sort=False,
    )
    if security_frames
    else pd.DataFrame(
        columns=[
            "security_id",
            "issuer_id",
            "issuer_name",
            "ticker",
            "exchange",
            "isin",
            "security_type",
            "is_adr",
            "source_region",
            "source_table",
        ]
    )
)

global_issuer_security_universe_df = (
    global_issuer_security_universe_df
    .dropna(
        subset=[
            "security_id",
        ]
    )
    .drop_duplicates(
        "security_id",
        keep="first",
    )
    .reset_index(drop=True)
)


global_entity_relationship_graph_df = (
    pd.concat(
        relationship_frames,
        ignore_index=True,
        sort=False,
    )
    if relationship_frames
    else pd.DataFrame(
        columns=[
            "from_entity_id",
            "to_entity_id",
            "relationship_type",
            "effective_start",
            "effective_end",
            "confidence",
            "source_region",
            "source_table",
        ]
    )
)

global_entity_relationship_graph_df = (
    global_entity_relationship_graph_df
    .drop_duplicates()
    .reset_index(drop=True)
)

regional_architecture_load_log_df = (
    pd.DataFrame(
        architecture_load_rows
    )
)

display(
    regional_architecture_load_log_df
)

print(
    "Global economic issuers:",
    len(
        global_economic_issuer_universe_df
    ),
)
print(
    "Global issuer securities:",
    len(
        global_issuer_security_universe_df
    ),
)
print(
    "Global entity relationships:",
    len(
        global_entity_relationship_graph_df
    ),
)


/tmp/ipykernel_1441/441425925.py:382: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(


,source_region,table_role,table_name,status,row_count,error
0,USA,issuer_universe,usa_economic_issuer_universe_df,LOADED,84,<NA>
1,USA,security_universe,usa_issuer_security_universe_df,LOADED,93,<NA>
2,USA,relationship_graph,usa_entity_relationship_graph_df,LOADED,0,<NA>
3,EUROPE,issuer_universe,europe_economic_issuer_universe_df,LOADED,77,<NA>
4,EUROPE,security_universe,europe_issuer_security_universe_df,LOADED,81,<NA>
5,EUROPE,relationship_graph,europe_entity_relationship_graph_df,LOADED,0,<NA>
6,JAPAN,issuer_universe,japan_economic_issuer_universe_df,LOADED,66,<NA>
7,JAPAN,security_universe,japan_issuer_security_universe_df,LOADED,67,<NA>
8,JAPAN,relationship_graph,japan_entity_relationship_graph_df,LOADED,0,<NA>
9,KOREA,issuer_universe,korea_economic_issuer_universe_df,LOADED,29,<NA>


Global economic issuers: 326
Global issuer securities: 318
Global entity relationships: 4


In [ ]:
# 5. BUILD THE UNION GLOBAL CONCEPT REGISTRY

concept_frames = []
concept_load_rows = []

CONCEPT_FIELDS = [
    "standard_concept",
    "statement_type",
    "expected_period_type",
    "expected_unit_family",
    "core_tier",
    "is_core",
    "aggregation_policy",
]


def harmonise_concept_dictionary(
    region,
    dataframe,
    source_table,
):
    output = pd.DataFrame(
        index=dataframe.index
    )

    output["standard_concept"] = (
        coalesce_columns(
            dataframe,
            [
                "standard_concept",
                "canonical_concept",
                "concept",
            ],
        )
    )

    for field in (
        set(CONCEPT_FIELDS)
        - {"standard_concept"}
    ):
        output[field] = coalesce_columns(
            dataframe,
            [field],
        )

    output["definition_source_region"] = (
        region
    )
    output["definition_source_table"] = (
        source_table
    )

    return output


for region in REGIONAL_MANIFESTS:
    (
        frame,
        table_name,
        status,
        error,
    ) = load_optional_alias(
        region,
        "concept_dictionary",
    )

    concept_load_rows.append({
        "source_region": region,
        "table_name": table_name,
        "status": status,
        "row_count": len(frame),
        "error": error,
    })

    if status == "LOADED":
        concept_frames.append(
            harmonise_concept_dictionary(
                region,
                frame,
                table_name,
            )
        )


regional_concept_dictionary_df = (
    pd.concat(
        concept_frames,
        ignore_index=True,
        sort=False,
    )
    if concept_frames
    else pd.DataFrame(
        columns=(
            CONCEPT_FIELDS
            + [
                "definition_source_region",
                "definition_source_table",
            ]
        )
    )
)

regional_concept_dictionary_df = (
    regional_concept_dictionary_df
    .dropna(
        subset=[
            "standard_concept",
        ]
    )
    .drop_duplicates()
    .reset_index(drop=True)
)

DEFINITION_REGION_PRIORITY = {
    "EUROPE": 1,
    "USA": 2,
    "JAPAN": 3,
    "KOREA": 4,
    "MAINLAND_CHINA": 5,
    "HONG_KONG": 6,
    "AUSTRALIA": 7,
}

regional_concept_dictionary_df[
    "_definition_priority"
] = (
    regional_concept_dictionary_df[
        "definition_source_region"
    ]
    .map(
        DEFINITION_REGION_PRIORITY
    )
    .fillna(99)
)

global_standard_concept_dictionary_df = (
    regional_concept_dictionary_df
    .sort_values(
        [
            "standard_concept",
            "_definition_priority",
        ]
    )
    .drop_duplicates(
        "standard_concept",
        keep="first",
    )
    .drop(
        columns=[
            "_definition_priority",
        ]
    )
    .reset_index(drop=True)
)

conflict_rows = []

for concept, group in (
    regional_concept_dictionary_df
    .groupby(
        "standard_concept",
        dropna=False,
    )
):
    for field in (
        set(CONCEPT_FIELDS)
        - {"standard_concept"}
    ):
        distinct = (
            group[field]
            .dropna()
            .astype("string")
            .drop_duplicates()
            .tolist()
        )

        if len(distinct) > 1:
            conflict_rows.append({
                "standard_concept": concept,
                "definition_field": field,
                "distinct_values": json.dumps(
                    distinct,
                    ensure_ascii=False,
                ),
                "source_regions": json.dumps(
                    sorted(
                        group[
                            "definition_source_region"
                        ]
                        .dropna()
                        .astype(str)
                        .unique()
                        .tolist()
                    ),
                    ensure_ascii=False,
                ),
            })


global_concept_definition_conflicts_df = (
    pd.DataFrame(
        conflict_rows,
        columns=[
            "standard_concept",
            "definition_field",
            "distinct_values",
            "source_regions",
        ],
    )
)

global_regional_concept_coverage_df = (
    regional_concept_dictionary_df
    .groupby(
        "definition_source_region",
        dropna=False,
    )
    .agg(
        concept_count=(
            "standard_concept",
            "nunique",
        ),
        definition_rows=(
            "standard_concept",
            "size",
        ),
    )
    .reset_index()
)

regional_concept_load_log_df = (
    pd.DataFrame(
        concept_load_rows
    )
)

print(
    "Global canonical concepts:",
    global_standard_concept_dictionary_df[
        "standard_concept"
    ].nunique(),
)

display(
    regional_concept_load_log_df
)
display(
    global_concept_definition_conflicts_df.head(
        100
    )
)


Global canonical concepts: 132


,source_region,table_name,status,row_count,error
0,USA,sec_standard_concept_dictionary_df,LOADED,147,<NA>
1,EUROPE,europe_standard_concept_dictionary_df,LOADED,111,<NA>
2,JAPAN,japan_standard_concept_dictionary_df,LOADED,91,<NA>
3,KOREA,korea_standard_concept_dictionary_df,LOADED,70,<NA>
4,HONG_KONG,hong_kong_standard_concept_dictionary_df,LOADED,90,<NA>
5,MAINLAND_CHINA,china_standard_concept_dictionary_df,LOADED,96,<NA>
6,AUSTRALIA,australia_standard_concept_dictionary_df,LOADED,69,<NA>


,standard_concept,definition_field,distinct_values,source_regions


In [ ]:
# 6. GLOBAL HARMONISED FACT SCHEMA AND FIELD REGISTRY

GLOBAL_FACT_COLUMNS = [
    "issuer_id",
    "security_id",
    "issuer_name",
    "ticker",
    "standard_concept",
    "statement_type",
    "period_type",
    "expected_period_type",
    "unit_family",
    "expected_unit_family",
    "period_start",
    "period_end",
    "fiscal_year",
    "fiscal_period",
    "report_type",
    "reporting_scope",
    "reported_value",
    "reported_currency",
    "unit_scale",
    "filing_id",
    "document_id",
    "document_content_sha256",
    "document_url",
    "form_type",
    "filing_date",
    "available_datetime",
    "available_date",
    "availability_basis",
    "source_system",
    "source_quality_score_regional",
    "mapping_priority",
    "selection_score_regional",
    "is_selected_standard_fact",
    "is_amendment",
    "amends_filing_id",
    "extraction_method",
    "source_line",
    "column_header",
    "parent_account_label",
    "data_quality_status_regional",
]

GENERIC_FIELD_CANDIDATES = {
    "issuer_id": [
        "issuer_id",
        "economic_issuer_id",
        "global_issuer_id",
    ],
    "security_id": [
        "security_id",
        "global_security_id",
    ],
    "issuer_name": [
        "issuer_name",
        "canonical_issuer_name",
        "company_name",
        "entity_name",
    ],
    "ticker": [
        "ticker",
        "asx_code",
        "symbol",
    ],
    "standard_concept": [
        "standard_concept",
        "canonical_concept",
    ],
    "statement_type": [
        "statement_type",
    ],
    "period_type": [
        "resolved_period_type",
        "period_type_inferred",
        "period_type",
    ],
    "expected_period_type": [
        "expected_period_type",
    ],
    "unit_family": [
        "unit_family",
        "reported_unit_family",
    ],
    "expected_unit_family": [
        "expected_unit_family",
    ],
    "period_start": [
        "period_start",
        "report_period_start",
    ],
    "period_end": [
        "resolved_period_end",
        "period_end_inferred",
        "report_period_end",
        "period_end",
    ],
    "fiscal_year": [
        "fiscal_year",
    ],
    "fiscal_period": [
        "fiscal_period",
    ],
    "report_type": [
        "report_type",
        "document_type",
    ],
    "reporting_scope": [
        "reporting_scope",
        "scope",
    ],
    "reported_value": [
        "canonical_value",
        "scaled_reported_value",
        "reported_value",
        "value",
    ],
    "reported_currency": [
        "reported_currency",
        "currency",
        "reporting_currency",
    ],
    "unit_scale": [
        "unit_scale",
        "unit_scale_inferred",
        "scale",
    ],
    "filing_id": [
        "source_filing_id",
        "filing_id",
        "accession_number",
        "document_id",
    ],
    "document_id": [
        "document_id",
        "source_filing_id",
        "accession_number",
    ],
    "document_content_sha256": [
        "document_content_sha256",
        "document_hash",
        "sha256",
    ],
    "document_url": [
        "document_url",
        "source_url",
        "filing_url",
    ],
    "form_type": [
        "form_type",
        "form",
        "sec_form",
    ],
    "filing_date": [
        "filing_date",
        "filed",
        "available_date",
    ],
    "available_datetime": [
        "available_datetime",
        "filing_datetime",
        "publication_datetime",
    ],
    "available_date": [
        "available_date",
        "filing_date",
        "publication_date",
    ],
    "availability_basis": [
        "availability_basis",
    ],
    "source_system": [
        "source_system",
        "filing_system",
        "source",
    ],
    "source_quality_score_regional": [
        "source_quality_score",
        "confidence_score",
    ],
    "mapping_priority": [
        "mapping_priority",
    ],
    "selection_score_regional": [
        "selection_score",
        "selection_score_regional",
    ],
    "is_selected_standard_fact": [
        "is_selected_standard_fact",
        "is_primary",
        "is_selected",
    ],
    "is_amendment": [
        "is_amendment",
    ],
    "amends_filing_id": [
        "amends_filing_id",
    ],
    "extraction_method": [
        "extraction_method",
        "mapping_method",
    ],
    "source_line": [
        "source_line",
        "source_context",
        "evidence_text",
    ],
    "column_header": [
        "column_header",
    ],
    "parent_account_label": [
        "parent_account_label",
    ],
    "data_quality_status_regional": [
        "data_quality_status",
        "quality_status",
    ],
}

DATETIME_FIELDS = [
    "period_start",
    "period_end",
    "filing_date",
    "available_datetime",
    "available_date",
]

print(
    "Global harmonised fields:",
    len(
        GLOBAL_FACT_COLUMNS
    ),
)


Global harmonised fields: 40


In [ ]:
# 7. SOURCE QUALITY AND PREFERRED ACCOUNTING-SOURCE RULES

SOURCE_QUALITY_RULES = pd.DataFrame([
    ("USA", r"SEC|EDGAR|COMPANY.?FACTS|XBRL", 100, "TIER_1_STRUCTURED_REGULATOR"),
    ("EUROPE", r"ESEF|XBRL|ESMA|FILING", 98, "TIER_1_STRUCTURED_REGULATOR"),
    ("JAPAN", r"EDINET|XBRL", 97, "TIER_1_STRUCTURED_REGULATOR"),
    ("KOREA", r"DART|OPENDART|XBRL", 97, "TIER_1_STRUCTURED_REGULATOR"),
    ("MAINLAND_CHINA", r"CNINFO", 92, "TIER_2_OFFICIAL_REGULATOR"),
    ("HONG_KONG", r"HKEX|HKEXNEWS", 90, "TIER_2_OFFICIAL_EXCHANGE"),
    ("AUSTRALIA", r"SEC_EDGAR", 100, "TIER_1_STRUCTURED_REGULATOR"),
    ("AUSTRALIA", r"ISSUER_INVESTOR_RELATIONS", 88, "TIER_3_OFFICIAL_ISSUER_DOCUMENT"),
    ("AUSTRALIA", r"LOCAL_USER_SUPPLIED_PDF", 76, "TIER_4_CONTROLLED_LOCAL_DOCUMENT"),
], columns=[
    "source_region",
    "source_system_pattern",
    "base_source_quality_score",
    "source_quality_tier",
])

EXTRACTION_METHOD_ADJUSTMENT = {
    "SEC_HTML_TABLE": 0,
    "XBRL": 0,
    "STRUCTURED_API": 0,
    "DETERMINISTIC_MAPPING": -2,
    "TEXT_LINE": -8,
    "REGEX": -8,
    "FULL_REGEX_MATCH": -5,
    "OCR": -18,
    "LLM": -15,
    "AI_ASSISTED": -15,
}

REGION_SOURCE_PRIORITY = {
    "USA": 1,
    "EUROPE": 1,
    "JAPAN": 1,
    "KOREA": 1,
    "MAINLAND_CHINA": 2,
    "HONG_KONG": 2,
    "AUSTRALIA": 3,
}

PREFERRED_SOURCE_DESCRIPTION = (
    "Issuer-level preferred accounting source is selected using the "
    "highest median harmonised source-quality score, followed by fact "
    "coverage and filing coverage. Secondary sources may contribute only "
    "incremental concepts or availability versions not supplied by the "
    "preferred source."
)

display(
    SOURCE_QUALITY_RULES
)


,source_region,source_system_pattern,base_source_quality_score,source_quality_tier
0,USA,SEC|EDGAR|COMPANY.?FACTS|XBRL,100,TIER_1_STRUCTURED_REGULATOR
1,EUROPE,ESEF|XBRL|ESMA|FILING,98,TIER_1_STRUCTURED_REGULATOR
2,JAPAN,EDINET|XBRL,97,TIER_1_STRUCTURED_REGULATOR
3,KOREA,DART|OPENDART|XBRL,97,TIER_1_STRUCTURED_REGULATOR
4,MAINLAND_CHINA,CNINFO,92,TIER_2_OFFICIAL_REGULATOR
5,HONG_KONG,HKEX|HKEXNEWS,90,TIER_2_OFFICIAL_EXCHANGE
6,AUSTRALIA,SEC_EDGAR,100,TIER_1_STRUCTURED_REGULATOR
7,AUSTRALIA,ISSUER_INVESTOR_RELATIONS,88,TIER_3_OFFICIAL_ISSUER_DOCUMENT
8,AUSTRALIA,LOCAL_USER_SUPPLIED_PDF,76,TIER_4_CONTROLLED_LOCAL_DOCUMENT


In [ ]:
# 8. HARMONISATION HELPERS

def infer_default_source_system(
    region,
):
    return {
        "USA": "SEC_EDGAR",
        "EUROPE": "ESEF",
        "JAPAN": "EDINET",
        "KOREA": "OPENDART",
        "HONG_KONG": "HKEXNEWS",
        "MAINLAND_CHINA": "CNINFO",
        "AUSTRALIA": "ISSUER_INVESTOR_RELATIONS",
    }.get(
        region,
        region,
    )


def normalise_reporting_scope(
    series,
):
    text = (
        series.astype("string")
        .fillna("")
        .str.upper()
    )

    return pd.Series(
        np.select(
            [
                text.str.contains(
                    r"CONSOLIDATED|CFS|GROUP",
                    regex=True,
                ),
                text.str.contains(
                    r"SEPARATE|OFS|PARENT",
                    regex=True,
                ),
            ],
            [
                "CONSOLIDATED",
                "SEPARATE",
            ],
            default="UNKNOWN",
        ),
        index=series.index,
        dtype="string",
    )


def harmonised_source_quality(
    region,
    source_system,
    extraction_method,
    regional_score,
):
    base_scores = []
    tiers = []

    rules = SOURCE_QUALITY_RULES[
        SOURCE_QUALITY_RULES[
            "source_region"
        ].eq(region)
    ]

    for value in (
        source_system
        .astype("string")
        .fillna("")
    ):
        matched = None

        for row in rules.itertuples(
            index=False
        ):
            if re.search(
                row.source_system_pattern,
                str(value),
                flags=re.IGNORECASE,
            ):
                matched = row
                break

        if matched is None:
            base_scores.append(65.0)
            tiers.append(
                "TIER_4_UNCLASSIFIED_OFFICIAL_SOURCE"
            )
        else:
            base_scores.append(
                float(
                    matched.base_source_quality_score
                )
            )
            tiers.append(
                matched.source_quality_tier
            )

    scores = pd.Series(
        base_scores,
        index=source_system.index,
        dtype="float64",
    )

    method_adjustment = (
        extraction_method
        .astype("string")
        .str.upper()
        .map(
            EXTRACTION_METHOD_ADJUSTMENT
        )
        .fillna(0)
        .astype(float)
    )

    scores = (
        scores
        + method_adjustment
    ).clip(
        lower=0,
        upper=100,
    )

    normalised_regional = (
        pd.to_numeric(
            regional_score,
            errors="coerce",
        )
    )

    # Regional scores may be expressed on either 0–1 or 0–100 scales.
    normalised_regional = np.where(
        normalised_regional.notna()
        & normalised_regional.le(1.0),
        normalised_regional * 100.0,
        normalised_regional,
    )

    normalised_regional = pd.Series(
        normalised_regional,
        index=scores.index,
        dtype="float64",
    )

    scores = (
        scores
        .combine(
            normalised_regional,
            lambda base, regional: (
                max(base, regional)
                if pd.notna(regional)
                else base
            ),
        )
        .clip(
            lower=0,
            upper=100,
        )
    )

    return (
        scores,
        pd.Series(
            tiers,
            index=scores.index,
            dtype="string",
        ),
    )


def harmonise_fact_table(
    region,
    dataframe,
    source_table,
    fact_layer,
):
    output = pd.DataFrame(
        index=dataframe.index
    )

    for field in GLOBAL_FACT_COLUMNS:
        output[field] = coalesce_columns(
            dataframe,
            GENERIC_FIELD_CANDIDATES.get(
                field,
                [field],
            ),
        )

    output["source_region"] = region
    output["source_table"] = (
        source_table
    )
    output["fact_layer"] = (
        fact_layer
    )
    output["source_row_number"] = (
        np.arange(
            len(output),
            dtype=np.int64,
        )
    )

    output["source_system"] = (
        output["source_system"]
        .astype("string")
        .fillna(
            infer_default_source_system(
                region
            )
        )
    )

    output["extraction_method"] = (
        output["extraction_method"]
        .astype("string")
    )

    (
        output["source_quality_score"],
        output["source_quality_tier"],
    ) = harmonised_source_quality(
        region,
        output["source_system"],
        output["extraction_method"],
        output[
            "source_quality_score_regional"
        ],
    )

    output["reporting_scope"] = (
        normalise_reporting_scope(
            output["reporting_scope"]
        )
    )

    for field in DATETIME_FIELDS:
        output[field] = pd.to_datetime(
            output[field],
            errors="coerce",
            utc=(
                field
                in {
                    "available_datetime",
                    "available_date",
                }
            ),
        )

    output["available_datetime"] = (
        output["available_datetime"]
        .combine_first(
            pd.to_datetime(
                output["available_date"],
                errors="coerce",
                utc=True,
            )
        )
    )

    output["available_date"] = (
        output["available_datetime"]
        .dt.normalize()
    )

    output["filing_date"] = (
        pd.to_datetime(
            output["filing_date"],
            errors="coerce",
        )
    )

    output["reported_value"] = (
        pd.to_numeric(
            output["reported_value"],
            errors="coerce",
        )
    )

    output["unit_scale"] = (
        pd.to_numeric(
            output["unit_scale"],
            errors="coerce",
        )
        .fillna(1.0)
    )

    output["fiscal_year"] = (
        pd.to_numeric(
            output["fiscal_year"],
            errors="coerce",
        )
        .astype("Int64")
    )

    output["is_amendment"] = (
        output["is_amendment"]
        .fillna(False)
        .astype("boolean")
    )

    output[
        "is_selected_standard_fact"
    ] = (
        output[
            "is_selected_standard_fact"
        ]
        .fillna(
            fact_layer
            == "STANDARDISED"
        )
        .astype("boolean")
    )

    output["document_id"] = (
        output["document_id"]
        .combine_first(
            output["filing_id"]
        )
    )

    output["filing_id"] = (
        output["filing_id"]
        .combine_first(
            output["document_id"]
        )
    )

    output = output.merge(
        global_standard_concept_dictionary_df[
            CONCEPT_FIELDS
        ],
        on="standard_concept",
        how="left",
        validate="m:1",
        suffixes=(
            "",
            "_canonical",
        ),
    )

    for field in [
        "statement_type",
        "expected_period_type",
        "expected_unit_family",
    ]:
        canonical = (
            f"{field}_canonical"
        )

        if canonical in output.columns:
            output[field] = (
                output[field]
                .combine_first(
                    output[canonical]
                )
            )

    output["period_type"] = (
        output["period_type"]
        .combine_first(
            output[
                "expected_period_type"
            ]
        )
    )

    output["unit_family"] = (
        output["unit_family"]
        .combine_first(
            output[
                "expected_unit_family"
            ]
        )
    )

    output["period_type_match"] = (
        output["period_type"]
        .astype("string")
        .eq(
            output[
                "expected_period_type"
            ].astype("string")
        )
    )

    output["unit_family_match"] = (
        output["unit_family"]
        .astype("string")
        .eq(
            output[
                "expected_unit_family"
            ].astype("string")
        )
    )

    output["is_numeric_fact"] = (
        output["reported_value"]
        .notna()
    )

    issuer_lookup = (
        global_economic_issuer_universe_df[
            [
                "issuer_id",
                "issuer_name",
            ]
        ]
        if not global_economic_issuer_universe_df.empty
        else pd.DataFrame(
            columns=[
                "issuer_id",
                "issuer_name",
            ]
        )
    )

    output = output.merge(
        issuer_lookup.rename(
            columns={
                "issuer_name": (
                    "issuer_name_authoritative"
                )
            }
        ),
        on="issuer_id",
        how="left",
        validate="m:1",
    )

    output["issuer_name"] = (
        output[
            "issuer_name_authoritative"
        ]
        .combine_first(
            output["issuer_name"]
        )
    )

    key_text = (
        output["source_region"]
        .astype("string")
        .fillna("")
        + "|"
        + output["source_table"]
        .astype("string")
        .fillna("")
        + "|"
        + output["source_row_number"]
        .astype("string")
        + "|"
        + output["filing_id"]
        .astype("string")
        .fillna("")
        + "|"
        + output["standard_concept"]
        .astype("string")
        .fillna("")
    )

    output[
        "global_source_observation_id"
    ] = key_text.map(
        lambda value: hashlib.sha256(
            value.encode()
        ).hexdigest()[:32]
    )

    return output


In [ ]:
# 9. LOAD AND HARMONISE REGIONAL STANDARDISED AND INCREMENTAL FACTS

standardised_frames = []
incremental_frames = []
fact_load_rows = []

for region in REGIONAL_MANIFESTS:
    for role, layer in [
        (
            "standardised_facts",
            "STANDARDISED",
        ),
        (
            "incremental_facts",
            "INCREMENTAL",
        ),
    ]:
        (
            frame,
            table_name,
            status,
            error,
        ) = load_optional_alias(
            region,
            role,
        )

        fact_load_rows.append({
            "source_region": region,
            "table_role": role,
            "table_name": table_name,
            "status": status,
            "row_count": len(frame),
            "error": error,
        })

        if status != "LOADED":
            continue

        harmonised = harmonise_fact_table(
            region,
            frame,
            table_name,
            layer,
        )

        if layer == "STANDARDISED":
            standardised_frames.append(
                harmonised
            )
        else:
            incremental_frames.append(
                harmonised
            )


global_fundamentals_combined_df = (
    pd.concat(
        standardised_frames,
        ignore_index=True,
        sort=False,
    )
    if standardised_frames
    else pd.DataFrame()
)

global_incremental_facts_df = (
    pd.concat(
        incremental_frames,
        ignore_index=True,
        sort=False,
    )
    if incremental_frames
    else pd.DataFrame(
        columns=(
            global_fundamentals_combined_df.columns
            if not global_fundamentals_combined_df.empty
            else GLOBAL_FACT_COLUMNS
        )
    )
)

regional_fact_load_log_df = (
    pd.DataFrame(
        fact_load_rows
    )
)

if global_fundamentals_combined_df.empty:
    raise RuntimeError(
        "No regional standardised-fundamentals table "
        "could be loaded."
    )

print(
    "Combined standardised fact rows:",
    len(
        global_fundamentals_combined_df
    ),
)
print(
    "Combined incremental fact rows:",
    len(
        global_incremental_facts_df
    ),
)

display(
    regional_fact_load_log_df
)


/tmp/ipykernel_1441/747136050.py:299: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_1441/747136050.py:299: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_1441/747136050.py:309: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(
/tmp/ipykernel_1441/747136050.py:299: FutureWarni

Combined standardised fact rows: 477374
Combined incremental fact rows: 8766


,source_region,table_role,table_name,status,row_count,error
0,USA,standardised_facts,sec_fundamentals_standardised_df,LOADED,387948,<NA>
1,USA,incremental_facts,None,TABLE_NOT_FOUND,0,<NA>
2,EUROPE,standardised_facts,europe_fundamentals_standardised_df,LOADED,11756,<NA>
3,EUROPE,incremental_facts,None,TABLE_NOT_FOUND,0,<NA>
4,JAPAN,standardised_facts,japan_fundamentals_standardised_df,LOADED,39814,<NA>
5,JAPAN,incremental_facts,None,TABLE_NOT_FOUND,0,<NA>
6,KOREA,standardised_facts,korea_fundamentals_standardised_df,LOADED,29678,<NA>
7,KOREA,incremental_facts,None,TABLE_NOT_FOUND,0,<NA>
8,HONG_KONG,standardised_facts,hong_kong_fundamentals_standardised_df,LOADED,534,<NA>
9,HONG_KONG,incremental_facts,None,TABLE_NOT_FOUND,0,<NA>


In [ ]:
# 10. VALIDATE UPSTREAM BLOCK 9 SYNONYM FEEDBACK

# Block 9 mapping decisions are now incorporated upstream:
#
#   Block 9 accepted synonym registry
#       -> Block 7 China / Hong Kong standardisation
#       -> Block 8 Australia standardisation
#       -> Block 10 Global combination
#
# Step 10 therefore validates that feedback contract. It does
# not replay review-queue decisions onto global fact rows.

REQUIRE_BLOCK_9_INTEGRATION = True

BLOCK9_FEEDBACK_METHOD = (
    "UPSTREAM_ACCEPTED_SYNONYM_REGISTRY"
)


# ------------------------------------------------------------
# 1. Generic manifest-table loader
# ------------------------------------------------------------

def normalise_manifest_table_name(
    value,
):
    text = str(
        value
        if value is not None
        else ""
    ).strip()

    if text.endswith("_df"):
        text = text[:-3]

    return text


def load_first_available_manifest_table(
    manifest_path,
    aliases,
):
    """
    Load the first available Parquet/CSV table from either:

    - a regional-style manifest using 'tables' and 'path'; or
    - a Block 9-style manifest using 'outputs' and 'saved_file'.

    Returns:
        frame,
        loaded_table_name,
        load_status,
        load_error
    """
    manifest_path = Path(
        manifest_path
    )

    if not manifest_path.exists():
        return (
            pd.DataFrame(),
            None,
            "MANIFEST_MISSING",
            f"Manifest not found: {manifest_path}",
        )

    try:
        payload = json.loads(
            manifest_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception as exc:
        return (
            pd.DataFrame(),
            None,
            "MANIFEST_READ_FAILED",
            repr(exc),
        )

    requested_aliases = [
        normalise_manifest_table_name(
            alias
        )
        for alias in aliases
    ]

    records = []

    for key in [
        "tables",
        "outputs",
    ]:
        if isinstance(
            payload.get(key),
            list,
        ):
            records.extend(
                payload[key]
            )

    # Defensive support for direct dictionary records.
    for value in payload.values():
        if not isinstance(value, dict):
            continue

        if any(
            key in value
            for key in [
                "table_name",
                "name",
                "path",
                "saved_file",
            ]
        ):
            records.append(value)

    candidates = []

    for record in records:
        if not isinstance(record, dict):
            continue

        table_name = (
            record.get("table_name")
            or record.get("name")
            or record.get("table")
        )

        normalised_name = (
            normalise_manifest_table_name(
                table_name
            )
        )

        if (
            normalised_name
            not in requested_aliases
        ):
            continue

        status = str(
            record.get(
                "status",
                "PASSED",
            )
        ).upper()

        if (
            status
            and not status.startswith("PASS")
            and status not in {
                "LOADED",
                "SUCCESS",
                "SUCCEEDED",
            }
        ):
            continue

        saved_path = (
            record.get("path")
            or record.get("saved_file")
            or record.get("file_path")
            or record.get("output_path")
        )

        if not saved_path:
            continue

        file_type = str(
            record.get(
                "file_type",
                "",
            )
        ).lower()

        candidates.append({
            "table_name":
                normalised_name,
            "path":
                Path(saved_path),
            "file_type":
                file_type,
            "alias_priority":
                requested_aliases.index(
                    normalised_name
                ),
        })

    # Filename fallback beside the manifest.
    if not candidates:
        for alias_priority, alias in enumerate(
            requested_aliases
        ):
            for suffix in [
                ".parquet",
                ".csv",
            ]:
                candidate_path = (
                    manifest_path.parent
                    / f"{alias}{suffix}"
                )

                if candidate_path.exists():
                    candidates.append({
                        "table_name":
                            alias,
                        "path":
                            candidate_path,
                        "file_type":
                            suffix.lstrip("."),
                        "alias_priority":
                            alias_priority,
                    })

    if not candidates:
        available_names = sorted({
            normalise_manifest_table_name(
                record.get(
                    "table_name",
                    record.get(
                        "name",
                        "",
                    ),
                )
            )
            for record in records
            if isinstance(record, dict)
        })

        return (
            pd.DataFrame(),
            None,
            "TABLE_NOT_FOUND",
            (
                f"Requested aliases: "
                f"{requested_aliases}. "
                f"Available sample: "
                f"{available_names[:40]}"
            ),
        )

    candidates = sorted(
        candidates,
        key=lambda item: (
            item["alias_priority"],
            (
                item["file_type"]
                != "parquet"
                and item["path"].suffix.lower()
                != ".parquet"
            ),
        ),
    )

    selected = candidates[0]
    selected_path = selected["path"]

    if not selected_path.exists():
        return (
            pd.DataFrame(),
            selected["table_name"],
            "FILE_NOT_FOUND",
            str(selected_path),
        )

    try:
        if (
            selected["file_type"] == "csv"
            or selected_path.suffix.lower()
            == ".csv"
        ):
            frame = pd.read_csv(
                selected_path,
                low_memory=False,
            )

        else:
            frame = pd.read_parquet(
                selected_path
            )

        return (
            frame,
            selected["table_name"],
            "LOADED",
            pd.NA,
        )

    except Exception as exc:
        return (
            pd.DataFrame(),
            selected["table_name"],
            "LOAD_FAILED",
            repr(exc),
        )


# ------------------------------------------------------------
# 2. Resolve regional manifest paths
# ------------------------------------------------------------

def resolve_regional_manifest_path(
    region_candidates,
    fallback_block_number,
):
    """
    Resolve a manifest from REGIONAL_MANIFESTS where possible,
    otherwise use the standard interim/block_N location.
    """
    if (
        "REGIONAL_MANIFESTS"
        in globals()
        and isinstance(
            REGIONAL_MANIFESTS,
            dict,
        )
    ):
        for key, value in (
            REGIONAL_MANIFESTS.items()
        ):
            key_text = str(
                key
            ).casefold()

            if any(
                candidate.casefold()
                in key_text
                for candidate
                in region_candidates
            ):
                return Path(value)

    return (
        DATA_ROOT
        / "interim"
        / f"block_{fallback_block_number}"
        / f"block_{fallback_block_number}_manifest.json"
    )


BLOCK_7_MANIFEST_PATH = (
    resolve_regional_manifest_path(
        [
            "hong",
            "china",
            "block_7",
        ],
        7,
    )
)

BLOCK_8_MANIFEST_PATH = (
    resolve_regional_manifest_path(
        [
            "australia",
            "asx",
            "block_8",
        ],
        8,
    )
)


# ------------------------------------------------------------
# 3. Preserve compatibility fields required downstream
# ------------------------------------------------------------

global_fact_rows_before_block9_validation = int(
    len(
        global_fundamentals_combined_df
    )
)

global_fundamentals_pre_block9_df = (
    global_fundamentals_combined_df.copy()
)


if (
    "standard_concept"
    not in global_fundamentals_combined_df.columns
):
    global_fundamentals_combined_df[
        "standard_concept"
    ] = pd.NA


if (
    "reported_value"
    not in global_fundamentals_combined_df.columns
):
    global_fundamentals_combined_df[
        "reported_value"
    ] = pd.NA


if (
    "reported_value_numeric"
    not in global_fundamentals_combined_df.columns
):
    global_fundamentals_combined_df[
        "reported_value_numeric"
    ] = pd.to_numeric(
        global_fundamentals_combined_df[
            "reported_value"
        ],
        errors="coerce",
    )


if (
    "original_standard_concept"
    not in global_fundamentals_combined_df.columns
):
    global_fundamentals_combined_df[
        "original_standard_concept"
    ] = global_fundamentals_combined_df[
        "standard_concept"
    ]


if (
    "effective_standard_concept"
    not in global_fundamentals_combined_df.columns
):
    global_fundamentals_combined_df[
        "effective_standard_concept"
    ] = global_fundamentals_combined_df[
        "standard_concept"
    ]


if (
    "original_reported_value"
    not in global_fundamentals_combined_df.columns
):
    global_fundamentals_combined_df[
        "original_reported_value"
    ] = global_fundamentals_combined_df[
        "reported_value"
    ]


if (
    "effective_reported_value"
    not in global_fundamentals_combined_df.columns
):
    global_fundamentals_combined_df[
        "effective_reported_value"
    ] = global_fundamentals_combined_df[
        "reported_value"
    ]


if (
    "original_reported_value_numeric"
    not in global_fundamentals_combined_df.columns
):
    global_fundamentals_combined_df[
        "original_reported_value_numeric"
    ] = global_fundamentals_combined_df[
        "reported_value_numeric"
    ]


# Compatibility fields formerly populated by the direct overlay.
compatibility_defaults = {
    "block9_acceptance_status":
        pd.NA,
    "block9_publication_status":
        pd.NA,
    "block9_mapping_basis":
        pd.NA,
    "block9_issue_signature":
        pd.NA,
    "block9_completion_method":
        pd.NA,
    "block9_ai_confidence":
        np.nan,
    "block9_decision_applied":
        False,
    "block9_decision_direct":
        False,
    "block9_decision_propagated":
        False,
}

for column, default_value in (
    compatibility_defaults.items()
):
    if (
        column
        not in global_fundamentals_combined_df.columns
    ):
        global_fundamentals_combined_df[
            column
        ] = default_value


for column in [
    "block9_decision_applied",
    "block9_decision_direct",
    "block9_decision_propagated",
]:
    global_fundamentals_combined_df[
        column
    ] = (
        global_fundamentals_combined_df[
            column
        ]
        .fillna(False)
        .astype(bool)
    )


# ------------------------------------------------------------
# 4. Load Block 9 accepted synonym registry
# ------------------------------------------------------------

(
    block_9_quality_control_outcomes_df,
    block_9_loaded_table_name,
    block_9_load_status,
    block_9_load_error,
) = load_first_available_manifest_table(
    BLOCK_9_MANIFEST_PATH,
    [
        "accepted_synonym_registry",
        "accepted_synonym_registry_df",
    ],
)


# ------------------------------------------------------------
# 5. Load Block 7 and Block 8 feedback evidence
# ------------------------------------------------------------

(
    hong_kong_block9_registry_matches_df,
    hong_kong_feedback_table_name,
    hong_kong_feedback_status,
    hong_kong_feedback_error,
) = load_first_available_manifest_table(
    BLOCK_7_MANIFEST_PATH,
    [
        "hong_kong_block9_registry_matches",
        "hong_kong_block9_registry_matches_df",
    ],
)


(
    china_block9_registry_matches_df,
    china_feedback_table_name,
    china_feedback_status,
    china_feedback_error,
) = load_first_available_manifest_table(
    BLOCK_7_MANIFEST_PATH,
    [
        "china_block9_registry_matches",
        "china_block9_registry_matches_df",
    ],
)


(
    australia_block9_registry_matches_df,
    australia_feedback_table_name,
    australia_feedback_status,
    australia_feedback_error,
) = load_first_available_manifest_table(
    BLOCK_8_MANIFEST_PATH,
    [
        "australia_block9_registry_matches",
        "australia_block9_registry_matches_df",
    ],
)


# ------------------------------------------------------------
# 6. Load refreshed regional production outputs for audit
# ------------------------------------------------------------

(
    hong_kong_refreshed_fundamentals_df,
    hong_kong_fundamentals_table_name,
    hong_kong_fundamentals_status,
    hong_kong_fundamentals_error,
) = load_first_available_manifest_table(
    BLOCK_7_MANIFEST_PATH,
    [
        "hong_kong_fundamentals_standardised",
        "hong_kong_fundamentals_standardised_df",
    ],
)


(
    china_refreshed_fundamentals_df,
    china_fundamentals_table_name,
    china_fundamentals_status,
    china_fundamentals_error,
) = load_first_available_manifest_table(
    BLOCK_7_MANIFEST_PATH,
    [
        "china_fundamentals_standardised",
        "china_fundamentals_standardised_df",
        "mainland_china_fundamentals_standardised",
        "mainland_china_fundamentals_standardised_df",
    ],
)


(
    australia_refreshed_fundamentals_df,
    australia_fundamentals_table_name,
    australia_fundamentals_status,
    australia_fundamentals_error,
) = load_first_available_manifest_table(
    BLOCK_8_MANIFEST_PATH,
    [
        "australia_fundamentals_standardised",
        "australia_fundamentals_standardised_df",
    ],
)


# ------------------------------------------------------------
# 7. Helper functions for regional audit
# ------------------------------------------------------------

def count_mapping_method_rows(
    frame,
    method="BLOCK9_ACCEPTED_SYNONYM",
):
    if (
        frame.empty
        or "mapping_method"
        not in frame.columns
    ):
        return 0

    return int(
        frame[
            "mapping_method"
        ]
        .astype("string")
        .eq(method)
        .fillna(False)
        .sum()
    )


def count_accepted_registry_rows(
    frame,
):
    if frame.empty:
        return 0

    if (
        "mapping_is_accepted"
        in frame.columns
    ):
        return int(
            frame[
                "mapping_is_accepted"
            ]
            .fillna(False)
            .astype(bool)
            .sum()
        )

    # The persisted match table is derived from an already accepted
    # Block 9 synonym registry. In the absence of a separate gate
    # column, all rows are accepted feedback evidence.
    return int(
        len(frame)
    )


def safe_unique_count(
    frame,
    column,
):
    if (
        frame.empty
        or column not in frame.columns
    ):
        return 0

    return int(
        frame[
            column
        ].nunique(
            dropna=True
        )
    )


def global_region_mask(
    frame,
    region_aliases,
):
    if (
        frame.empty
        or "source_region"
        not in frame.columns
    ):
        return pd.Series(
            False,
            index=frame.index,
            dtype=bool,
        )

    aliases = {
        str(alias).casefold()
        for alias in region_aliases
    }

    return (
        frame[
            "source_region"
        ]
        .astype("string")
        .str.strip()
        .str.casefold()
        .isin(aliases)
        .fillna(False)
    )


# ------------------------------------------------------------
# 8. Build regional feedback validation report
# ------------------------------------------------------------

regional_contracts = [
    {
        "source_region":
            "HONG_KONG",
        "global_region_aliases": [
            "HONG_KONG",
            "Hong Kong",
        ],
        "feedback_frame":
            hong_kong_block9_registry_matches_df,
        "feedback_status":
            hong_kong_feedback_status,
        "feedback_error":
            hong_kong_feedback_error,
        "production_frame":
            hong_kong_refreshed_fundamentals_df,
        "production_status":
            hong_kong_fundamentals_status,
        "production_error":
            hong_kong_fundamentals_error,
        "expected_source_table":
            "hong_kong_fundamentals_standardised_df",
    },
    {
        "source_region":
            "MAINLAND_CHINA",
        "global_region_aliases": [
            "MAINLAND_CHINA",
            "Mainland China",
            "China",
        ],
        "feedback_frame":
            china_block9_registry_matches_df,
        "feedback_status":
            china_feedback_status,
        "feedback_error":
            china_feedback_error,
        "production_frame":
            china_refreshed_fundamentals_df,
        "production_status":
            china_fundamentals_status,
        "production_error":
            china_fundamentals_error,
        "expected_source_table":
            "china_fundamentals_standardised_df",
    },
    {
        "source_region":
            "AUSTRALIA",
        "global_region_aliases": [
            "AUSTRALIA",
            "Australia",
        ],
        "feedback_frame":
            australia_block9_registry_matches_df,
        "feedback_status":
            australia_feedback_status,
        "feedback_error":
            australia_feedback_error,
        "production_frame":
            australia_refreshed_fundamentals_df,
        "production_status":
            australia_fundamentals_status,
        "production_error":
            australia_fundamentals_error,
        "expected_source_table":
            "australia_fundamentals_standardised_df",
    },
]


regional_feedback_rows = []

for contract in regional_contracts:
    feedback_frame = contract[
        "feedback_frame"
    ]

    production_frame = contract[
        "production_frame"
    ]

    global_mask = global_region_mask(
        global_fundamentals_combined_df,
        contract[
            "global_region_aliases"
        ],
    )

    global_region_df = (
        global_fundamentals_combined_df.loc[
            global_mask
        ]
    )

    regional_feedback_rows.append({
        "source_region":
            contract[
                "source_region"
            ],
        "feedback_load_status":
            contract[
                "feedback_status"
            ],
        "feedback_load_error":
            contract[
                "feedback_error"
            ],
        "registry_match_rows":
            len(
                feedback_frame
            ),
        "accepted_registry_match_rows":
            count_accepted_registry_rows(
                feedback_frame
            ),
        "production_load_status":
            contract[
                "production_status"
            ],
        "production_load_error":
            contract[
                "production_error"
            ],
        "regional_standardised_rows":
            len(
                production_frame
            ),
        "regional_block9_provenance_rows":
            count_mapping_method_rows(
                production_frame
            ),
        "regional_unique_issuers":
            safe_unique_count(
                production_frame,
                "issuer_id",
            ),
        "regional_unique_documents":
            safe_unique_count(
                production_frame,
                "document_id",
            ),
        "global_combined_rows":
            len(
                global_region_df
            ),
        "global_unique_issuers":
            safe_unique_count(
                global_region_df,
                "issuer_id",
            ),
        "global_unique_documents":
            safe_unique_count(
                global_region_df,
                "document_id",
            ),
        "global_source_table_present": (
            int(
                global_region_df[
                    "source_table"
                ]
                .astype("string")
                .eq(
                    contract[
                        "expected_source_table"
                    ]
                )
                .sum()
            )
            > 0
            if (
                not global_region_df.empty
                and "source_table"
                in global_region_df.columns
            )
            else False
        ),
    })


block_9_feedback_by_region_df = (
    pd.DataFrame(
        regional_feedback_rows
    )
)


block_9_feedback_by_region_df[
    "feedback_contract_passed"
] = (
    block_9_feedback_by_region_df[
        "feedback_load_status"
    ].eq("LOADED")
    & block_9_feedback_by_region_df[
        "production_load_status"
    ].eq("LOADED")
    & block_9_feedback_by_region_df[
        "registry_match_rows"
    ].gt(0)
    & block_9_feedback_by_region_df[
        "accepted_registry_match_rows"
    ].gt(0)
    & block_9_feedback_by_region_df[
        "regional_standardised_rows"
    ].gt(0)
    & block_9_feedback_by_region_df[
        "global_combined_rows"
    ].gt(0)
    & block_9_feedback_by_region_df[
        "global_source_table_present"
    ].fillna(False)
)


# ------------------------------------------------------------
# 9. Validate permanent global observation IDs
# ------------------------------------------------------------

if (
    "global_source_observation_id"
    not in global_fundamentals_combined_df.columns
):
    raise RuntimeError(
        "global_fundamentals_combined_df does not contain "
        "global_source_observation_id."
    )


global_observation_id_non_null_rows = int(
    global_fundamentals_combined_df[
        "global_source_observation_id"
    ]
    .notna()
    .sum()
)

global_observation_id_unique_values = int(
    global_fundamentals_combined_df[
        "global_source_observation_id"
    ]
    .nunique(
        dropna=True
    )
)

global_observation_id_duplicate_rows = int(
    global_fundamentals_combined_df[
        "global_source_observation_id"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)


if (
    global_observation_id_non_null_rows
    != len(
        global_fundamentals_combined_df
    )
):
    raise RuntimeError(
        "Some global rows have no "
        "global_source_observation_id."
    )


if global_observation_id_duplicate_rows > 0:
    raise RuntimeError(
        "global_source_observation_id is not unique."
    )


# ------------------------------------------------------------
# 10. Empty retrospective-overlay audit
# ------------------------------------------------------------

block_9_overlay_audit_df = pd.DataFrame(
    columns=[
        "global_source_observation_id",
        "original_standard_concept",
        "effective_standard_concept",
        "original_reported_value",
        "effective_reported_value",
        "overlay_status",
        "integration_method",
    ]
)


global_fact_rows_updated = 0
concept_rows_updated = 0
value_rows_updated = 0
direct_rows_updated = 0
propagated_rows_updated = 0


# ------------------------------------------------------------
# 11. Step 10 summary
# ------------------------------------------------------------

regional_contracts_passed = int(
    block_9_feedback_by_region_df[
        "feedback_contract_passed"
    ]
    .fillna(False)
    .sum()
)

regional_contracts_expected = int(
    len(
        block_9_feedback_by_region_df
    )
)

accepted_registry_match_rows_total = int(
    block_9_feedback_by_region_df[
        "accepted_registry_match_rows"
    ]
    .sum()
)

regional_provenance_rows_total = int(
    block_9_feedback_by_region_df[
        "regional_block9_provenance_rows"
    ]
    .sum()
)


block_9_quality_control_summary_df = pd.DataFrame({
    "metric": [
        "integration_method",
        "block9_manifest_exists",
        "block9_registry_table_loaded",
        "block9_registry_load_status",
        "block9_registry_load_error",
        "block9_registry_rows",
        "regional_feedback_contracts_expected",
        "regional_feedback_contracts_passed",
        "accepted_registry_match_rows_total",
        "regional_provenance_rows_total",
        "global_fact_rows_before_validation",
        "global_fact_rows_after_validation",
        "global_source_observation_id_non_null_rows",
        "global_source_observation_id_unique_values",
        "global_source_observation_id_duplicate_rows",
        "global_fact_rows_updated_in_block10",
        "concept_rows_updated_in_block10",
        "value_rows_updated_in_block10",
        "direct_rows_updated_in_block10",
        "propagated_rows_updated_in_block10",
        "overlay_audit_rows",
    ],
    "value": [
        BLOCK9_FEEDBACK_METHOD,
        BLOCK_9_MANIFEST_PATH.exists(),
        block_9_loaded_table_name,
        block_9_load_status,
        block_9_load_error,
        len(
            block_9_quality_control_outcomes_df
        ),
        regional_contracts_expected,
        regional_contracts_passed,
        accepted_registry_match_rows_total,
        regional_provenance_rows_total,
        global_fact_rows_before_block9_validation,
        len(
            global_fundamentals_combined_df
        ),
        global_observation_id_non_null_rows,
        global_observation_id_unique_values,
        global_observation_id_duplicate_rows,
        global_fact_rows_updated,
        concept_rows_updated,
        value_rows_updated,
        direct_rows_updated,
        propagated_rows_updated,
        len(
            block_9_overlay_audit_df
        ),
    ],
})


print("=" * 88)
print(
    "BLOCK 10 — UPSTREAM BLOCK 9 FEEDBACK VALIDATION"
)
print("=" * 88)

print(
    "Integration method:",
    BLOCK9_FEEDBACK_METHOD,
)

print(
    "Block 9 registry:",
    block_9_load_status,
    f"({len(block_9_quality_control_outcomes_df):,} rows)",
)

print(
    "Regional feedback contracts passed:",
    f"{regional_contracts_passed}/"
    f"{regional_contracts_expected}",
)

print(
    "Global fact rows:",
    f"{len(global_fundamentals_combined_df):,}",
)

print(
    "Unique global observation IDs:",
    f"{global_observation_id_unique_values:,}",
)

print(
    "Retrospective rows modified in Block 10:",
    "0",
)


display(
    block_9_feedback_by_region_df
)

display(
    block_9_quality_control_summary_df
)


# ------------------------------------------------------------
# 12. Mandatory validation gates
# ------------------------------------------------------------

validation_failures = []


if (
    block_9_load_status
    != "LOADED"
):
    validation_failures.append(
        (
            "Block 9 accepted synonym registry did not load. "
            f"Status={block_9_load_status}; "
            f"detail={block_9_load_error}"
        )
    )


failed_regions_df = (
    block_9_feedback_by_region_df.loc[
        ~block_9_feedback_by_region_df[
            "feedback_contract_passed"
        ]
        .fillna(False)
    ]
    .copy()
)


if not failed_regions_df.empty:
    failed_region_names = (
        failed_regions_df[
            "source_region"
        ]
        .astype("string")
        .tolist()
    )

    validation_failures.append(
        (
            "The upstream Block 9 feedback contract failed "
            "for: "
            + ", ".join(
                failed_region_names
            )
        )
    )


if (
    global_fact_rows_before_block9_validation
    != len(
        global_fundamentals_combined_df
    )
):
    validation_failures.append(
        "Validation-only Step 10 changed the global row count."
    )


if global_fact_rows_updated != 0:
    validation_failures.append(
        "Validation-only Step 10 modified global facts."
    )


if (
    REQUIRE_BLOCK_9_INTEGRATION
    and validation_failures
):
    display(
        failed_regions_df
    )

    raise RuntimeError(
        "Block 10 upstream Block 9 feedback validation failed:\n- "
        + "\n- ".join(
            validation_failures
        )
    )


print(
    "\nBlock 10 upstream Block 9 feedback validation passed."
)

print(
    "Accepted AI mappings were applied in Blocks 7 and 8 "
    "before the regional production facts entered Block 10."
)

BLOCK 10 — UPSTREAM BLOCK 9 FEEDBACK VALIDATION
Integration method: UPSTREAM_ACCEPTED_SYNONYM_REGISTRY
Block 9 registry: LOADED (531 rows)
Regional feedback contracts passed: 3/3
Global fact rows: 477,374
Unique global observation IDs: 477,374
Retrospective rows modified in Block 10: 0


,source_region,feedback_load_status,feedback_load_error,registry_match_rows,accepted_registry_match_rows,production_load_status,production_load_error,regional_standardised_rows,regional_block9_provenance_rows,regional_unique_issuers,regional_unique_documents,global_combined_rows,global_unique_issuers,global_unique_documents,global_source_table_present,feedback_contract_passed
0,HONG_KONG,LOADED,<NA>,60,60,LOADED,<NA>,534,98,16,72,534,16,72,True,True
1,MAINLAND_CHINA,LOADED,<NA>,173,173,LOADED,<NA>,6985,256,4,197,6985,4,197,True,True
2,AUSTRALIA,LOADED,<NA>,35,35,LOADED,<NA>,659,78,5,45,659,5,45,True,True


,metric,value
0,integration_method,UPSTREAM_ACCEPTED_SYNONYM_REGISTRY
1,block9_manifest_exists,True
2,block9_registry_table_loaded,accepted_synonym_registry
3,block9_registry_load_status,LOADED
4,block9_registry_load_error,<NA>
5,block9_registry_rows,531
6,regional_feedback_contracts_expected,3
7,regional_feedback_contracts_passed,3
8,accepted_registry_match_rows_total,268
9,regional_provenance_rows_total,432



Block 10 upstream Block 9 feedback validation passed.
Accepted AI mappings were applied in Blocks 7 and 8 before the regional production facts entered Block 10.


In [ ]:
# 11. PREFERRED ACCOUNTING SOURCES AND CROSS-REGION PRECEDENCE

source_summary_df = (
    global_fundamentals_combined_df
    .groupby(
        [
            "issuer_id",
            "source_region",
            "source_system",
        ],
        dropna=False,
    )
    .agg(
        median_source_quality=(
            "source_quality_score",
            "median",
        ),
        standardised_fact_rows=(
            "standard_concept",
            "size",
        ),
        standard_concepts=(
            "standard_concept",
            "nunique",
        ),
        filing_count=(
            "filing_id",
            "nunique",
        ),
        latest_available_datetime=(
            "available_datetime",
            "max",
        ),
    )
    .reset_index()
)

source_summary_df[
    "region_source_priority"
] = (
    source_summary_df[
        "source_region"
    ]
    .map(
        REGION_SOURCE_PRIORITY
    )
    .fillna(99)
)

source_summary_df[
    "preferred_source_rank"
] = (
    source_summary_df
    .sort_values(
        [
            "issuer_id",
            "median_source_quality",
            "standard_concepts",
            "filing_count",
            "region_source_priority",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            True,
        ],
        na_position="last",
    )
    .groupby(
        "issuer_id",
        dropna=False,
    )
    .cumcount()
    + 1
)

global_preferred_accounting_source_df = (
    source_summary_df[
        source_summary_df[
            "preferred_source_rank"
        ].eq(1)
    ]
    .copy()
    .rename(
        columns={
            "source_region": (
                "preferred_source_region"
            ),
            "source_system": (
                "preferred_source_system"
            ),
            "median_source_quality": (
                "preferred_source_quality"
            ),
        }
    )
    .reset_index(drop=True)
)

global_preferred_accounting_source_df[
    "selection_basis"
] = (
    "QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE"
)

global_fundamentals_combined_df = (
    global_fundamentals_combined_df
    .merge(
        global_preferred_accounting_source_df[
            [
                "issuer_id",
                "preferred_source_region",
                "preferred_source_system",
                "preferred_source_quality",
            ]
        ],
        on="issuer_id",
        how="left",
        validate="m:1",
    )
)

global_fundamentals_combined_df[
    "is_preferred_accounting_source"
] = (
    global_fundamentals_combined_df[
        "source_region"
    ].eq(
        global_fundamentals_combined_df[
            "preferred_source_region"
        ]
    )
    & global_fundamentals_combined_df[
        "source_system"
    ].eq(
        global_fundamentals_combined_df[
            "preferred_source_system"
        ]
    )
)


def stable_date_key(
    series,
):
    return (
        pd.to_datetime(
            series,
            errors="coerce",
        )
        .dt.strftime(
            "%Y-%m-%d"
        )
        .fillna(
            "UNKNOWN_DATE"
        )
    )


def stable_string(
    series,
    fallback="",
):
    return (
        series.astype("string")
        .fillna(fallback)
    )


facts = global_fundamentals_combined_df

facts["preferred_source_bonus"] = (
    facts[
        "is_preferred_accounting_source"
    ]
    .fillna(False)
    .astype(int)
    * 25
)

facts["consolidated_scope_bonus"] = (
    facts["reporting_scope"]
    .eq("CONSOLIDATED")
    .astype(int)
    * 10
)

facts["period_match_bonus"] = (
    facts["period_type_match"]
    .fillna(False)
    .astype(int)
    * 6
)

facts["unit_match_bonus"] = (
    facts["unit_family_match"]
    .fillna(False)
    .astype(int)
    * 6
)

facts["numeric_bonus"] = (
    facts["is_numeric_fact"]
    .fillna(False)
    .astype(int)
    * 5
)

facts["regional_primary_bonus"] = (
    facts[
        "is_selected_standard_fact"
    ]
    .fillna(False)
    .astype(int)
    * 5
)

facts["global_precedence_score"] = (
    facts[
        "source_quality_score"
    ].fillna(0)
    + facts[
        "preferred_source_bonus"
    ]
    + facts[
        "consolidated_scope_bonus"
    ]
    + facts[
        "period_match_bonus"
    ]
    + facts[
        "unit_match_bonus"
    ]
    + facts[
        "numeric_bonus"
    ]
    + facts[
        "regional_primary_bonus"
    ]
    + facts[
        "is_amendment"
    ]
    .fillna(False)
    .astype(int)
    * 2
)

facts["economic_fact_key"] = (
    stable_string(
        facts["issuer_id"]
    )
    + "|"
    + stable_string(
        facts["standard_concept"]
    )
    + "|"
    + stable_date_key(
        facts["period_start"]
    )
    + "|"
    + stable_date_key(
        facts["period_end"]
    )
    + "|"
    + stable_string(
        facts["period_type"],
        "UNKNOWN_PERIOD_TYPE",
    )
    + "|"
    + stable_string(
        facts["reporting_scope"],
        "UNKNOWN_SCOPE",
    )
    + "|"
    + stable_string(
        facts["reported_currency"],
        "UNKNOWN_CURRENCY",
    )
)

facts["availability_version_key"] = (
    facts["economic_fact_key"]
    + "|"
    + pd.to_datetime(
        facts[
            "available_datetime"
        ],
        errors="coerce",
        utc=True,
    )
    .dt.strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )
    .fillna(
        "UNKNOWN_AVAILABILITY"
    )
)

facts[
    "document_deduplication_key"
] = (
    stable_string(
        facts[
            "document_content_sha256"
        ]
    )
    .where(
        facts[
            "document_content_sha256"
        ].notna(),
        stable_string(
            facts["document_id"]
        )
        .where(
            facts[
                "document_id"
            ].notna(),
            stable_string(
                facts["filing_id"]
            ),
        ),
    )
)

facts = facts.sort_values(
    [
        "availability_version_key",
        "global_precedence_score",
        "available_datetime",
        "filing_date",
        "global_source_observation_id",
    ],
    ascending=[
        True,
        False,
        False,
        False,
        True,
    ],
    na_position="last",
)

facts[
    "is_global_selected_candidate"
] = ~facts.duplicated(
    "availability_version_key",
    keep="first",
)

global_fundamentals_combined_df = (
    facts.reset_index(drop=True)
)

global_source_precedence_df = (
    SOURCE_QUALITY_RULES.copy()
)

display(
    global_preferred_accounting_source_df.head(
        100
    )
)


,issuer_id,preferred_source_region,preferred_source_system,preferred_source_quality,standardised_fact_rows,standard_concepts,filing_count,latest_available_datetime,region_source_priority,preferred_source_rank,selection_basis
0,GAI_00575497A3B780818B41,USA,SEC_EDGAR,100.0,511,40,10,2026-05-14 10:55:37+00:00,1,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE
1,GAI_022FF6951192C2C9FF42,USA,SEC_EDGAR,100.0,8368,62,65,2026-07-17 13:35:42+00:00,1,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE
2,GAI_0323BDA6A082569EF596,USA,SEC_EDGAR,100.0,2677,54,21,2026-05-28 20:09:32+00:00,1,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE
3,GAI_044EBD9F4107A05EB8ED,USA,SEC_EDGAR,100.0,1463,50,9,2026-04-17 10:12:40+00:00,1,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE
4,GAI_054CDD5CF4D68DA7EA0C,KOREA,OPENDART,97.0,1946,52,22,2026-03-11 15:00:00+00:00,1,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE
5,GAI_09DA7F36C135795A0916,MAINLAND_CHINA,CNINFO,92.0,1731,74,50,2026-04-29 16:00:00+00:00,2,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE
6,GAI_0AB747BD45FC8DEC2F1C,USA,SEC_EDGAR,100.0,2906,60,24,2026-05-21 12:24:32+00:00,1,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE
7,GAI_0BC5F5DC59A9F6E7A189,USA,SEC_EDGAR,100.0,957,30,10,2026-04-16 10:05:49+00:00,1,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE
8,GAI_108E1BDC6D93C3DCD73A,USA,SEC_EDGAR,100.0,8425,63,64,2026-05-04 20:14:02+00:00,1,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE
9,GAI_118982B3D7A3654914FD,HONG_KONG,HKEXNEWS,90.0,4,4,1,2020-03-30 14:14:00+00:00,2,1,QUALITY_THEN_CONCEPT_AND_FILING_COVERAGE


In [ ]:
# 12. SELECTED FACTS, ALTERNATIVES, INCREMENTAL SUPPLEMENTS AND RESOLUTION LOG

global_fundamentals_selected_df = (
    global_fundamentals_combined_df[
        global_fundamentals_combined_df[
            "is_global_selected_candidate"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

global_fundamentals_alternatives_df = (
    global_fundamentals_combined_df[
        ~global_fundamentals_combined_df[
            "is_global_selected_candidate"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

selected_lookup_df = (
    global_fundamentals_selected_df[
        [
            "availability_version_key",
            "global_source_observation_id",
            "source_region",
            "source_system",
            "global_precedence_score",
            "reported_value",
        ]
    ]
    .rename(
        columns={
            "global_source_observation_id": (
                "selected_observation_id"
            ),
            "source_region": (
                "selected_source_region"
            ),
            "source_system": (
                "selected_source_system"
            ),
            "global_precedence_score": (
                "selected_precedence_score"
            ),
            "reported_value": (
                "selected_reported_value"
            ),
        }
    )
)

global_duplicate_resolution_df = (
    global_fundamentals_combined_df
    .merge(
        selected_lookup_df,
        on="availability_version_key",
        how="left",
        validate="m:1",
    )
)

global_duplicate_resolution_df[
    "resolution_status"
] = np.where(
    global_duplicate_resolution_df[
        "is_global_selected_candidate"
    ],
    "SELECTED",
    "ALTERNATIVE_REJECTED",
)

global_duplicate_resolution_df[
    "precedence_score_gap"
] = (
    global_duplicate_resolution_df[
        "selected_precedence_score"
    ]
    - global_duplicate_resolution_df[
        "global_precedence_score"
    ]
)

global_duplicate_resolution_df[
    "resolution_reason"
] = np.select(
    [
        global_duplicate_resolution_df[
            "is_global_selected_candidate"
        ],
        ~global_duplicate_resolution_df[
            "is_preferred_accounting_source"
        ].fillna(False),
        global_duplicate_resolution_df[
            "source_quality_score"
        ]
        < global_duplicate_resolution_df
        .groupby(
            "availability_version_key"
        )[
            "source_quality_score"
        ]
        .transform("max"),
        ~global_duplicate_resolution_df[
            "period_type_match"
        ].fillna(False),
        ~global_duplicate_resolution_df[
            "unit_family_match"
        ].fillna(False),
    ],
    [
        "HIGHEST_GLOBAL_PRECEDENCE",
        "NON_PREFERRED_ACCOUNTING_SOURCE",
        "LOWER_SOURCE_QUALITY",
        "PERIOD_TYPE_MISMATCH",
        "UNIT_FAMILY_MISMATCH",
    ],
    default="LOWER_COMPOSITE_PRECEDENCE",
)


if not global_incremental_facts_df.empty:
    selected_identity = set(
        global_fundamentals_selected_df[
            [
                "issuer_id",
                "standard_concept",
                "period_end",
                "available_datetime",
            ]
        ]
        .astype("string")
        .fillna("")
        .agg("|".join, axis=1)
    )

    incremental_identity = (
        global_incremental_facts_df[
            [
                "issuer_id",
                "standard_concept",
                "period_end",
                "available_datetime",
            ]
        ]
        .astype("string")
        .fillna("")
        .agg("|".join, axis=1)
    )

    global_incremental_facts_df[
        "supplements_selected_facts"
    ] = ~incremental_identity.isin(
        selected_identity
    )

    global_incremental_facts_selected_df = (
        global_incremental_facts_df[
            global_incremental_facts_df[
                "supplements_selected_facts"
            ]
        ]
        .copy()
        .reset_index(drop=True)
    )

else:
    global_incremental_facts_selected_df = (
        global_incremental_facts_df.copy()
    )

print(
    "Globally selected facts:",
    len(
        global_fundamentals_selected_df
    ),
)
print(
    "Alternatives retained:",
    len(
        global_fundamentals_alternatives_df
    ),
)
print(
    "Selected incremental supplements:",
    len(
        global_incremental_facts_selected_df
    ),
)


Globally selected facts: 459215
Alternatives retained: 18159
Selected incremental supplements: 0


In [ ]:
# 13. GLOBAL FILING METADATA AND VERSION HISTORY

filing_frames = []
filing_load_rows = []

FILING_FIELDS = [
    "issuer_id",
    "security_id",
    "issuer_name",
    "ticker",
    "filing_id",
    "document_id",
    "document_content_sha256",
    "document_url",
    "form_type",
    "report_type",
    "period_end",
    "filing_date",
    "available_datetime",
    "available_date",
    "availability_basis",
    "reporting_scope",
    "is_amendment",
    "amends_filing_id",
    "source_system",
]


for region in REGIONAL_MANIFESTS:
    (
        frame,
        table_name,
        status,
        error,
    ) = load_optional_alias(
        region,
        "filings",
    )

    filing_load_rows.append({
        "source_region": region,
        "table_name": table_name,
        "status": status,
        "row_count": len(frame),
        "error": error,
    })

    if status != "LOADED":
        continue

    output = pd.DataFrame(
        index=frame.index
    )

    for field in FILING_FIELDS:
        output[field] = coalesce_columns(
            frame,
            GENERIC_FIELD_CANDIDATES.get(
                field,
                [field],
            ),
        )

    output["source_region"] = region
    output["source_table"] = (
        table_name
    )

    output["source_system"] = (
        output["source_system"]
        .astype("string")
        .fillna(
            infer_default_source_system(
                region
            )
        )
    )

    output["available_datetime"] = (
        pd.to_datetime(
            output["available_datetime"],
            errors="coerce",
            utc=True,
        )
        .combine_first(
            pd.to_datetime(
                output["available_date"],
                errors="coerce",
                utc=True,
            )
        )
    )

    output["available_date"] = (
        output["available_datetime"]
        .dt.normalize()
    )

    output["period_end"] = (
        pd.to_datetime(
            output["period_end"],
            errors="coerce",
        )
    )

    output["filing_date"] = (
        pd.to_datetime(
            output["filing_date"],
            errors="coerce",
        )
    )

    output["reporting_scope"] = (
        normalise_reporting_scope(
            output["reporting_scope"]
        )
    )

    output["filing_id"] = (
        output["filing_id"]
        .combine_first(
            output["document_id"]
        )
    )

    output["document_id"] = (
        output["document_id"]
        .combine_first(
            output["filing_id"]
        )
    )

    filing_frames.append(
        output
    )


global_filing_metadata_df = (
    pd.concat(
        filing_frames,
        ignore_index=True,
        sort=False,
    )
    if filing_frames
    else pd.DataFrame(
        columns=(
            FILING_FIELDS
            + [
                "source_region",
                "source_table",
            ]
        )
    )
)

global_filing_metadata_df = (
    global_filing_metadata_df
    .drop_duplicates(
        [
            "source_region",
            "filing_id",
            "document_id",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

global_filing_load_log_df = (
    pd.DataFrame(
        filing_load_rows
    )
)


global_fundamentals_selected_df = (
    global_fundamentals_selected_df
    .sort_values(
        [
            "economic_fact_key",
            "available_datetime",
            "filing_date",
            "global_source_observation_id",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

global_fundamentals_selected_df[
    "fact_version_number"
] = (
    global_fundamentals_selected_df
    .groupby(
        "economic_fact_key",
        dropna=False,
    )
    .cumcount()
    + 1
)

global_fundamentals_selected_df[
    "previous_reported_value"
] = (
    global_fundamentals_selected_df
    .groupby(
        "economic_fact_key",
        dropna=False,
    )[
        "reported_value"
    ]
    .shift(1)
)

global_fundamentals_selected_df[
    "value_revision"
] = (
    global_fundamentals_selected_df[
        "reported_value"
    ]
    - global_fundamentals_selected_df[
        "previous_reported_value"
    ]
)

global_fundamentals_selected_df[
    "is_value_revision"
] = (
    global_fundamentals_selected_df[
        "previous_reported_value"
    ].notna()
    & global_fundamentals_selected_df[
        "reported_value"
    ].notna()
    & ~np.isclose(
        global_fundamentals_selected_df[
            "reported_value"
        ],
        global_fundamentals_selected_df[
            "previous_reported_value"
        ],
        equal_nan=True,
    )
)

global_fact_version_history_df = (
    global_fundamentals_selected_df[
        [
            "economic_fact_key",
            "availability_version_key",
            "fact_version_number",
            "issuer_id",
            "security_id",
            "standard_concept",
            "period_start",
            "period_end",
            "period_type",
            "available_datetime",
            "reported_value",
            "previous_reported_value",
            "value_revision",
            "is_value_revision",
            "source_region",
            "source_system",
            "filing_id",
            "document_id",
            "is_amendment",
        ]
    ]
    .copy()
)

display(
    global_filing_load_log_df
)


/tmp/ipykernel_1441/2724876256.py:187: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  result = result.combine_first(
/tmp/ipykernel_1441/2724876256.py:187: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  result = result.combine_first(
/tmp/ipykernel_1441/3499921228.py:121: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  .combine_first(
/tmp/ip

,source_region,table_name,status,row_count,error
0,USA,sec_filing_metadata_df,LOADED,9898,<NA>
1,EUROPE,europe_filing_metadata_df,LOADED,10281,<NA>
2,JAPAN,japan_filing_metadata_df,LOADED,678,<NA>
3,KOREA,korea_filing_metadata_df,LOADED,322,<NA>
4,HONG_KONG,hong_kong_filing_metadata_df,LOADED,90,<NA>
5,MAINLAND_CHINA,china_disclosure_inventory_df,LOADED,210,<NA>
6,AUSTRALIA,australia_filing_metadata_df,LOADED,70,<NA>


In [ ]:
# 14. POINT-IN-TIME HELPERS AND MONTHLY LONG PANEL

def latest_global_fundamentals_as_of(
    as_of_date,
    *,
    issuer_ids=None,
    security_ids=None,
    standard_concepts=None,
):
    cutoff = pd.Timestamp(
        as_of_date
    )

    if cutoff.tzinfo is None:
        cutoff = cutoff.tz_localize(
            "UTC"
        )
    else:
        cutoff = cutoff.tz_convert(
            "UTC"
        )

    result = (
        global_fundamentals_selected_df[
            pd.to_datetime(
                global_fundamentals_selected_df[
                    "available_datetime"
                ],
                errors="coerce",
                utc=True,
            )
            <= cutoff
        ]
        .copy()
    )

    if issuer_ids is not None:
        result = result[
            result["issuer_id"].isin(
                set(
                    issuer_ids
                )
            )
        ]

    if security_ids is not None:
        result = result[
            result["security_id"].isin(
                set(
                    security_ids
                )
            )
        ]

    if standard_concepts is not None:
        result = result[
            result[
                "standard_concept"
            ].isin(
                set(
                    standard_concepts
                )
            )
        ]

    return (
        result
        .sort_values(
            [
                "economic_fact_key",
                "available_datetime",
                "fact_version_number",
                "global_precedence_score",
            ],
            na_position="last",
        )
        .drop_duplicates(
            "economic_fact_key",
            keep="last",
        )
        .reset_index(drop=True)
    )


research_month_ends = pd.date_range(
    PANEL_START_DATE,
    PANEL_END_DATE,
    freq=MONTH_END_FREQUENCY,
)

monthly_panel_frames = []

for research_date in tqdm(
    research_month_ends,
    desc="Building monthly point-in-time fundamentals",
):
    snapshot = latest_global_fundamentals_as_of(
        research_date,
        standard_concepts=(
            CORE_INTERPRETATION_CONCEPTS
        ),
    )

    if snapshot.empty:
        continue

    snapshot["research_date"] = (
        research_date
    )

    snapshot[
        "fundamental_age_days"
    ] = (
        research_date.tz_localize(
            "UTC"
        )
        - pd.to_datetime(
            snapshot[
                "available_datetime"
            ],
            errors="coerce",
            utc=True,
        )
    ).dt.days

    snapshot[
        "fundamental_age_months"
    ] = (
        snapshot[
            "fundamental_age_days"
        ]
        / 30.4375
    )

    snapshot[
        "maximum_staleness_days"
    ] = np.where(
        snapshot[
            "expected_period_type"
        ]
        .astype("string")
        .str.upper()
        .isin(
            {
                "INSTANT",
                "BALANCE",
            }
        ),
        INSTANT_MAX_STALENESS_DAYS,
        FLOW_MAX_STALENESS_DAYS,
    )

    snapshot["is_stale"] = (
        snapshot[
            "fundamental_age_days"
        ]
        > snapshot[
            "maximum_staleness_days"
        ]
    )

    snapshot[
        "availability_lag_days"
    ] = (
        pd.to_datetime(
            snapshot[
                "available_datetime"
            ],
            errors="coerce",
            utc=True,
        ).dt.tz_localize(None)
        - pd.to_datetime(
            snapshot["period_end"],
            errors="coerce",
        )
    ).dt.days

    monthly_panel_frames.append(
        snapshot
    )


global_monthly_fundamental_panel_df = (
    pd.concat(
        monthly_panel_frames,
        ignore_index=True,
        sort=False,
    )
    if monthly_panel_frames
    else pd.DataFrame()
)

if not global_monthly_fundamental_panel_df.empty:
    global_monthly_fundamental_panel_df[
        "research_usable_value"
    ] = (
        global_monthly_fundamental_panel_df[
            "reported_value"
        ]
        .where(
            ~global_monthly_fundamental_panel_df[
                "is_stale"
            ]
        )
    )

print(
    "Monthly point-in-time fact rows:",
    len(
        global_monthly_fundamental_panel_df
    ),
)


Building monthly point-in-time fundamentals:   0%|          | 0/81 [00:00<?, ?it/s]

Monthly point-in-time fact rows: 3180784


In [ ]:
# 15. DERIVED GENERIC INTERPRETATION FEATURES

if global_monthly_fundamental_panel_df.empty:
    global_monthly_accounting_wide_df = pd.DataFrame()
    global_monthly_interpretation_features_df = pd.DataFrame()

else:
    usable_long_df = (
        global_monthly_fundamental_panel_df[
            ~global_monthly_fundamental_panel_df[
                "is_stale"
            ]
        ]
        .copy()
    )

    global_monthly_accounting_wide_df = (
        usable_long_df
        .pivot_table(
            index=[
                "research_date",
                "issuer_id",
            ],
            columns=(
                "standard_concept"
            ),
            values=(
                "research_usable_value"
            ),
            aggfunc="last",
        )
        .reset_index()
    )

    global_monthly_accounting_wide_df.columns.name = None

    features = (
        global_monthly_accounting_wide_df
        .copy()
    )

    def safe_divide(
        numerator,
        denominator,
    ):
        result = (
            pd.to_numeric(
                numerator,
                errors="coerce",
            )
            / pd.to_numeric(
                denominator,
                errors="coerce",
            )
        )

        return result.replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )

    def column_or_nan(
        frame,
        column,
    ):
        if column in frame.columns:
            return pd.to_numeric(
                frame[column],
                errors="coerce",
            )

        return pd.Series(
            np.nan,
            index=frame.index,
            dtype="float64",
        )

    revenue = column_or_nan(
        features,
        "revenue",
    )
    gross_profit = column_or_nan(
        features,
        "gross_profit",
    )
    operating_income = column_or_nan(
        features,
        "operating_income",
    )
    ebitda = column_or_nan(
        features,
        "ebitda",
    )
    net_income = column_or_nan(
        features,
        "net_income",
    )
    cash = column_or_nan(
        features,
        "cash_and_cash_equivalents",
    )
    inventory = column_or_nan(
        features,
        "inventory",
    )
    assets = column_or_nan(
        features,
        "total_assets",
    )
    liabilities = column_or_nan(
        features,
        "total_liabilities",
    )
    equity = column_or_nan(
        features,
        "total_equity",
    )
    debt = column_or_nan(
        features,
        "total_debt",
    )
    operating_cash_flow = column_or_nan(
        features,
        "operating_cash_flow",
    )
    capex = column_or_nan(
        features,
        "capital_expenditure",
    )
    research_development = column_or_nan(
        features,
        "research_development",
    )
    intangible_assets = column_or_nan(
        features,
        "intangible_assets",
    )

    features["gross_margin"] = (
        safe_divide(
            gross_profit,
            revenue,
        )
    )

    features["operating_margin"] = (
        safe_divide(
            operating_income,
            revenue,
        )
    )

    features["ebitda_margin"] = (
        safe_divide(
            ebitda,
            revenue,
        )
    )

    features["net_margin"] = (
        safe_divide(
            net_income,
            revenue,
        )
    )

    features["return_on_assets"] = (
        safe_divide(
            net_income,
            assets,
        )
    )

    features["return_on_equity"] = (
        safe_divide(
            net_income,
            equity,
        )
    )

    features["asset_turnover"] = (
        safe_divide(
            revenue,
            assets,
        )
    )

    features[
        "inventory_intensity"
    ] = safe_divide(
        inventory,
        assets,
    )

    features[
        "capital_intensity"
    ] = safe_divide(
        capex.abs(),
        revenue.abs(),
    )

    features[
        "research_development_intensity"
    ] = safe_divide(
        research_development.abs(),
        revenue.abs(),
    )

    features[
        "intangible_intensity"
    ] = safe_divide(
        intangible_assets,
        assets,
    )

    features["cash_ratio"] = (
        safe_divide(
            cash,
            liabilities,
        )
    )

    features["leverage"] = (
        safe_divide(
            debt,
            assets,
        )
    )

    features["net_debt"] = (
        debt
        - cash
    )

    features[
        "operating_cash_conversion"
    ] = safe_divide(
        operating_cash_flow,
        net_income,
    )

    features["accruals_ratio"] = (
        safe_divide(
            net_income
            - operating_cash_flow,
            assets,
        )
    )

    features = features.sort_values(
        [
            "issuer_id",
            "research_date",
        ]
    )

    growth_columns = {
        "revenue": "revenue_growth",
        "net_income": "earnings_growth",
        "total_assets": "asset_growth",
        "capital_expenditure": "capex_growth",
        "research_development": (
            "research_development_growth"
        ),
    }

    for source_column, feature_column in (
        growth_columns.items()
    ):
        if source_column in features.columns:
            features[feature_column] = (
                features
                .groupby(
                    "issuer_id",
                    dropna=False,
                )[source_column]
                .pct_change(
                    periods=12,
                    fill_method=None,
                )
            )
        else:
            features[feature_column] = (
                np.nan
            )

    feature_columns = [
        "gross_margin",
        "operating_margin",
        "ebitda_margin",
        "net_margin",
        "return_on_assets",
        "return_on_equity",
        "asset_turnover",
        "inventory_intensity",
        "capital_intensity",
        "research_development_intensity",
        "intangible_intensity",
        "cash_ratio",
        "leverage",
        "net_debt",
        "operating_cash_conversion",
        "accruals_ratio",
        "revenue_growth",
        "earnings_growth",
        "asset_growth",
        "capex_growth",
        "research_development_growth",
    ]

    global_monthly_interpretation_features_df = (
        features[
            [
                "research_date",
                "issuer_id",
            ]
            + feature_columns
        ]
        .copy()
        .reset_index(drop=True)
    )

    global_monthly_interpretation_features_df = (
        global_monthly_interpretation_features_df
        .merge(
            global_economic_issuer_universe_df[
                [
                    "issuer_id",
                    "issuer_name",
                    "country_of_incorporation",
                    "reporting_currency",
                ]
            ],
            on="issuer_id",
            how="left",
            validate="m:1",
        )
    )

print(
    "Monthly interpretation feature rows:",
    len(
        global_monthly_interpretation_features_df
    ),
)


Monthly interpretation feature rows: 10711


In [ ]:
# 16. FEATURE, ISSUER AND MONTHLY COVERAGE REPORTS

global_region_coverage_df = (
    global_fundamentals_selected_df
    .groupby(
        [
            "source_region",
            "source_quality_tier",
        ],
        dropna=False,
    )
    .agg(
        selected_fact_rows=(
            "reported_value",
            "size",
        ),
        issuer_count=(
            "issuer_id",
            "nunique",
        ),
        security_count=(
            "security_id",
            "nunique",
        ),
        concept_count=(
            "standard_concept",
            "nunique",
        ),
        filing_count=(
            "filing_id",
            "nunique",
        ),
        earliest_period=(
            "period_end",
            "min",
        ),
        latest_period=(
            "period_end",
            "max",
        ),
        latest_available=(
            "available_datetime",
            "max",
        ),
    )
    .reset_index()
)

global_concept_coverage_df = (
    global_standard_concept_dictionary_df
    .merge(
        global_fundamentals_selected_df
        .groupby(
            "standard_concept",
            dropna=False,
        )
        .agg(
            selected_fact_rows=(
                "reported_value",
                "size",
            ),
            issuer_count=(
                "issuer_id",
                "nunique",
            ),
            security_count=(
                "security_id",
                "nunique",
            ),
            region_count=(
                "source_region",
                "nunique",
            ),
            filing_count=(
                "filing_id",
                "nunique",
            ),
        )
        .reset_index(),
        on="standard_concept",
        how="left",
    )
)

issuer_summary_df = (
    global_fundamentals_selected_df
    .groupby(
        "issuer_id",
        dropna=False,
    )
    .agg(
        security_count=(
            "security_id",
            "nunique",
        ),
        region_count=(
            "source_region",
            "nunique",
        ),
        concept_count=(
            "standard_concept",
            "nunique",
        ),
        selected_fact_rows=(
            "reported_value",
            "size",
        ),
        filing_count=(
            "filing_id",
            "nunique",
        ),
        earliest_period=(
            "period_end",
            "min",
        ),
        latest_period=(
            "period_end",
            "max",
        ),
        latest_available=(
            "available_datetime",
            "max",
        ),
    )
    .reset_index()
)

global_issuer_coverage_df = (
    global_economic_issuer_universe_df
    .merge(
        issuer_summary_df,
        on="issuer_id",
        how="left",
        validate="1:1",
    )
)

if global_monthly_interpretation_features_df.empty:
    global_feature_coverage_df = pd.DataFrame()
    global_issuer_feature_coverage_df = pd.DataFrame()
    global_monthly_feature_coverage_df = pd.DataFrame()

else:
    non_identifier_columns = [
        column
        for column in (
            global_monthly_interpretation_features_df.columns
        )
        if column not in {
            "research_date",
            "issuer_id",
            "issuer_name",
            "country_of_incorporation",
            "reporting_currency",
        }
    ]

    feature_coverage_rows = []

    for feature in non_identifier_columns:
        series = (
            global_monthly_interpretation_features_df[
                feature
            ]
        )

        feature_coverage_rows.append({
            "feature": feature,
            "observed_rows": int(
                series.notna().sum()
            ),
            "total_rows": len(
                series
            ),
            "coverage_ratio": float(
                series.notna().mean()
            ),
            "issuer_count": int(
                global_monthly_interpretation_features_df
                .loc[
                    series.notna(),
                    "issuer_id",
                ]
                .nunique()
            ),
            "first_observed_date": (
                global_monthly_interpretation_features_df
                .loc[
                    series.notna(),
                    "research_date",
                ]
                .min()
            ),
            "last_observed_date": (
                global_monthly_interpretation_features_df
                .loc[
                    series.notna(),
                    "research_date",
                ]
                .max()
            ),
        })

    global_feature_coverage_df = pd.DataFrame(
        feature_coverage_rows
    )

    issuer_feature_rows = []

    for issuer_id, group in (
        global_monthly_interpretation_features_df
        .groupby(
            "issuer_id",
            dropna=False,
        )
    ):
        for feature in non_identifier_columns:
            issuer_feature_rows.append({
                "issuer_id": issuer_id,
                "feature": feature,
                "observed_rows": int(
                    group[feature].notna().sum()
                ),
                "total_rows": len(group),
                "coverage_ratio": float(
                    group[feature].notna().mean()
                ),
            })

    global_issuer_feature_coverage_df = (
        pd.DataFrame(
            issuer_feature_rows
        )
        .merge(
            global_economic_issuer_universe_df[
                [
                    "issuer_id",
                    "issuer_name",
                ]
            ],
            on="issuer_id",
            how="left",
            validate="m:1",
        )
    )

    monthly_feature_rows = []

    for research_date, group in (
        global_monthly_interpretation_features_df
        .groupby(
            "research_date",
            dropna=False,
        )
    ):
        for feature in non_identifier_columns:
            monthly_feature_rows.append({
                "research_date": research_date,
                "feature": feature,
                "observed_issuers": int(
                    group.loc[
                        group[feature].notna(),
                        "issuer_id",
                    ].nunique()
                ),
                "eligible_issuers": int(
                    group["issuer_id"].nunique()
                ),
                "coverage_ratio": float(
                    group[feature].notna().mean()
                ),
            })

    global_monthly_feature_coverage_df = (
        pd.DataFrame(
            monthly_feature_rows
        )
    )

display(
    global_region_coverage_df
)
display(
    global_feature_coverage_df
)


,source_region,source_quality_tier,selected_fact_rows,issuer_count,security_count,concept_count,filing_count,earliest_period,latest_period,latest_available
0,AUSTRALIA,TIER_1_STRUCTURED_REGULATOR,190,2,2,42,7,2023-12-31,2025-12-31,2026-04-29 00:00:00+00:00
1,AUSTRALIA,TIER_3_OFFICIAL_ISSUER_DOCUMENT,406,3,3,40,34,NaT,NaT,2026-04-23 23:00:00+00:00
2,EUROPE,TIER_1_STRUCTURED_REGULATOR,11276,30,0,69,139,2019-01-01,2026-04-01,2026-06-11 17:15:32.267243+00:00
3,HONG_KONG,TIER_2_OFFICIAL_EXCHANGE,534,16,16,39,72,NaT,NaT,2026-04-30 09:10:00+00:00
4,JAPAN,TIER_1_STRUCTURED_REGULATOR,39754,27,27,46,519,2013-03-31,2026-03-31,2026-06-26 02:44:00+00:00
5,KOREA,TIER_1_STRUCTURED_REGULATOR,15771,11,0,68,294,2019-03-31,2026-03-31,2026-03-17 15:00:00+00:00
6,MAINLAND_CHINA,TIER_2_OFFICIAL_REGULATOR,6016,4,4,84,190,NaT,NaT,2026-04-29 16:00:00+00:00
7,USA,TIER_1_STRUCTURED_REGULATOR,385268,84,84,84,3552,2005-10-29,2026-07-10,2026-07-17 13:35:42+00:00


,feature,observed_rows,total_rows,coverage_ratio,issuer_count,first_observed_date,last_observed_date
0,gross_margin,8072,10711,0.753618,127,2019-12-31,2026-08-31
1,operating_margin,9363,10711,0.874148,146,2019-12-31,2026-08-31
2,ebitda_margin,0,10711,0.000000,0,NaT,NaT
3,net_margin,10032,10711,0.936607,163,2019-12-31,2026-08-31
4,return_on_assets,10464,10711,0.976940,158,2019-12-31,2026-08-31
5,return_on_equity,10377,10711,0.968817,158,2019-12-31,2026-08-31
6,asset_turnover,9990,10711,0.932686,156,2019-12-31,2026-08-31
7,inventory_intensity,7427,10711,0.693399,120,2019-12-31,2026-08-31
8,capital_intensity,5228,10711,0.488096,79,2019-12-31,2026-08-31
9,research_development_intensity,31,10711,0.002894,1,2024-02-29,2026-08-31


In [ ]:
# 17. DATA-QUALITY FLAGS AND RESEARCH-USABILITY CLASSIFICATION

selected = (
    global_fundamentals_selected_df
    .copy()
)

selected[
    "dq_missing_issuer_id"
] = selected["issuer_id"].isna()

selected[
    "dq_missing_period_end"
] = selected["period_end"].isna()

selected[
    "dq_missing_available_datetime"
] = selected[
    "available_datetime"
].isna()

selected[
    "dq_missing_reported_value"
] = selected[
    "reported_value"
].isna()

selected[
    "dq_period_type_mismatch"
] = ~selected[
    "period_type_match"
].fillna(False)

selected[
    "dq_unit_family_mismatch"
] = ~selected[
    "unit_family_match"
].fillna(False)

selected[
    "dq_low_source_quality"
] = (
    selected[
        "source_quality_score"
    ].fillna(0)
    < 70
)

selected[
    "dq_unknown_reporting_scope"
] = selected[
    "reporting_scope"
].eq("UNKNOWN")

selected[
    "dq_missing_currency_for_monetary_fact"
] = (
    selected[
        "expected_unit_family"
    ]
    .astype("string")
    .str.upper()
    .eq("MONETARY")
    & selected[
        "reported_currency"
    ].isna()
)

DQ_COLUMNS = [
    "dq_missing_issuer_id",
    "dq_missing_period_end",
    "dq_missing_available_datetime",
    "dq_missing_reported_value",
    "dq_period_type_mismatch",
    "dq_unit_family_mismatch",
    "dq_low_source_quality",
    "dq_unknown_reporting_scope",
    "dq_missing_currency_for_monetary_fact",
]

selected[
    "data_quality_issue_count"
] = (
    selected[DQ_COLUMNS]
    .fillna(False)
    .astype(int)
    .sum(axis=1)
)

selected[
    "research_usability"
] = np.select(
    [
        (
            selected[
                "data_quality_issue_count"
            ].eq(0)
            & selected[
                "source_quality_score"
            ].ge(90)
        ),
        (
            selected[
                "data_quality_issue_count"
            ].le(1)
            & selected[
                "source_quality_score"
            ].ge(70)
        ),
        (
            selected[
                "data_quality_issue_count"
            ].le(2)
        ),
    ],
    [
        "INTERPRETATION_READY",
        "USABLE_WITH_MISSINGNESS_CONTROL",
        "ROBUSTNESS_ONLY",
    ],
    default=(
        "EXCLUDE_PENDING_REVIEW"
    ),
)

global_fundamentals_selected_df = (
    selected
)

global_research_usability_report_df = (
    global_fundamentals_selected_df
    .groupby(
        [
            "source_region",
            "research_usability",
        ],
        dropna=False,
    )
    .agg(
        fact_rows=(
            "reported_value",
            "size",
        ),
        issuer_count=(
            "issuer_id",
            "nunique",
        ),
        concept_count=(
            "standard_concept",
            "nunique",
        ),
        average_quality_issues=(
            "data_quality_issue_count",
            "mean",
        ),
    )
    .reset_index()
)

global_quality_report_df = pd.DataFrame({
    "metric": [
        "selected_fact_rows",
        "selected_issuers",
        "selected_securities",
        "selected_concepts",
        "missing_issuer_id_rows",
        "missing_period_end_rows",
        "missing_available_datetime_rows",
        "missing_value_rows",
        "period_type_mismatch_rows",
        "unit_family_mismatch_rows",
        "low_source_quality_rows",
        "interpretation_ready_rows",
    ],
    "value": [
        len(
            global_fundamentals_selected_df
        ),
        global_fundamentals_selected_df[
            "issuer_id"
        ].nunique(),
        global_fundamentals_selected_df[
            "security_id"
        ].nunique(),
        global_fundamentals_selected_df[
            "standard_concept"
        ].nunique(),
        int(
            global_fundamentals_selected_df[
                "dq_missing_issuer_id"
            ].sum()
        ),
        int(
            global_fundamentals_selected_df[
                "dq_missing_period_end"
            ].sum()
        ),
        int(
            global_fundamentals_selected_df[
                "dq_missing_available_datetime"
            ].sum()
        ),
        int(
            global_fundamentals_selected_df[
                "dq_missing_reported_value"
            ].sum()
        ),
        int(
            global_fundamentals_selected_df[
                "dq_period_type_mismatch"
            ].sum()
        ),
        int(
            global_fundamentals_selected_df[
                "dq_unit_family_mismatch"
            ].sum()
        ),
        int(
            global_fundamentals_selected_df[
                "dq_low_source_quality"
            ].sum()
        ),
        int(
            global_fundamentals_selected_df[
                "research_usability"
            ].eq(
                "INTERPRETATION_READY"
            ).sum()
        ),
    ],
})

display(
    global_research_usability_report_df
)
display(
    global_quality_report_df
)


,source_region,research_usability,fact_rows,issuer_count,concept_count,average_quality_issues
0,AUSTRALIA,EXCLUDE_PENDING_REVIEW,596,5,47,3.697987
1,EUROPE,EXCLUDE_PENDING_REVIEW,3,1,2,3.000000
2,EUROPE,ROBUSTNESS_ONLY,10678,30,66,2.000000
3,EUROPE,USABLE_WITH_MISSINGNESS_CONTROL,595,28,4,1.000000
4,HONG_KONG,ROBUSTNESS_ONLY,534,16,39,2.000000
5,JAPAN,EXCLUDE_PENDING_REVIEW,225,25,21,3.000000
6,JAPAN,ROBUSTNESS_ONLY,38299,27,45,2.000000
7,JAPAN,USABLE_WITH_MISSINGNESS_CONTROL,1230,27,1,1.000000
8,KOREA,EXCLUDE_PENDING_REVIEW,291,11,17,3.079038
9,KOREA,ROBUSTNESS_ONLY,8080,11,66,2.000000


,metric,value
0,selected_fact_rows,459215
1,selected_issuers,168
2,selected_securities,131
3,selected_concepts,113
4,missing_issuer_id_rows,19427
5,missing_period_end_rows,6956
6,missing_available_datetime_rows,8098
7,missing_value_rows,334
8,period_type_mismatch_rows,1136
9,unit_family_mismatch_rows,206


In [ ]:
# 18. LATENT-FACTOR AND QUANTCONNECT EXPORTS

def release_unused_memory():
    """
    Request Python garbage collection and, where available,
    return unused heap memory to the Colab runtime.
    """
    gc.collect()

    try:
        ctypes.CDLL(
            "libc.so.6"
        ).malloc_trim(0)

    except Exception:
        pass


release_unused_memory()

# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def existing_columns(
    frame: pd.DataFrame,
    requested_columns,
):
    return [
        column
        for column in requested_columns
        if column in frame.columns
    ]


def first_existing_column(
    frame: pd.DataFrame,
    candidates,
):
    return next(
        (
            column
            for column in candidates
            if column in frame.columns
        ),
        None,
    )


def safe_nunique(
    frame: pd.DataFrame,
    column: str,
) -> int:
    if (
        frame.empty
        or column not in frame.columns
    ):
        return 0

    return int(
        frame[column].nunique(
            dropna=True
        )
    )


# ------------------------------------------------------------
# 2. Compact latent-factor model matrix
# ------------------------------------------------------------

latent_identifier_columns = existing_columns(
    global_monthly_interpretation_features_df,
    [
        "research_date",
        "issuer_id",
        "security_id",
        "global_issuer_id",
        "source_region",
        "research_usability",
    ],
)

latent_numeric_feature_columns = [
    column
    for column in (
        global_monthly_interpretation_features_df
        .select_dtypes(
            include=[np.number]
        )
        .columns
    )
    if column not in {
        "research_date",
    }
]

latent_factor_export_columns = list(
    dict.fromkeys(
        latent_identifier_columns
        + latent_numeric_feature_columns
    )
)

# Create only the compact modelling table, rather than copying
# every diagnostic and source-text column.
latent_factor_model_matrix_df = (
    global_monthly_interpretation_features_df.loc[
        :,
        latent_factor_export_columns,
    ]
    .drop_duplicates(
        existing_columns(
            global_monthly_interpretation_features_df,
            [
                "research_date",
                "issuer_id",
            ],
        ),
        keep="last",
    )
    .reset_index(drop=True)
)

if (
    "research_date"
    in latent_factor_model_matrix_df.columns
):
    latent_factor_model_matrix_df[
        "research_date"
    ] = pd.to_datetime(
        latent_factor_model_matrix_df[
            "research_date"
        ],
        errors="coerce",
    )

latent_factor_model_matrix_df = (
    latent_factor_model_matrix_df
    .sort_values(
        existing_columns(
            latent_factor_model_matrix_df,
            [
                "research_date",
                "issuer_id",
            ],
        )
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 3. Latent-factor feature dictionary
# ------------------------------------------------------------

latent_factor_feature_dictionary_df = (
    pd.DataFrame({
        "feature_name": (
            latent_numeric_feature_columns
        ),
        "feature_group": (
            "ACCOUNTING_INTERPRETATION"
        ),
        "point_in_time": True,
        "missing_values_are_zero": False,
        "recommended_preprocessing": (
            "cross_sectional_winsorise_"
            "then_standardise"
        ),
        "quantconnect_safe": True,
    })
)


# ------------------------------------------------------------
# 4. Compact QuantConnect security map
# ------------------------------------------------------------

security_source_df = (
    global_issuer_security_universe_df
)

security_id_column = first_existing_column(
    security_source_df,
    [
        "security_id",
        "global_security_id",
    ],
)

symbol_column = first_existing_column(
    security_source_df,
    [
        "quantconnect_symbol",
        "primary_ticker",
        "ticker",
        "security_ticker",
        "symbol",
    ],
)

market_column = first_existing_column(
    security_source_df,
    [
        "quantconnect_market",
        "market",
        "exchange_code",
        "primary_exchange",
    ],
)

security_map_columns = existing_columns(
    security_source_df,
    [
        "issuer_id",
        "global_issuer_id",
        security_id_column,
        symbol_column,
        market_column,
        "security_name",
        "security_type",
        "listing_currency",
        "country_code",
        "is_primary_listing",
        "is_active",
    ],
)

quantconnect_security_map_df = (
    security_source_df.loc[
        :,
        list(
            dict.fromkeys(
                security_map_columns
            )
        ),
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

security_rename_map = {}

if security_id_column is not None:
    security_rename_map[
        security_id_column
    ] = "security_id"

if symbol_column is not None:
    security_rename_map[
        symbol_column
    ] = "quantconnect_symbol"

if market_column is not None:
    security_rename_map[
        market_column
    ] = "quantconnect_market"

quantconnect_security_map_df = (
    quantconnect_security_map_df.rename(
        columns=security_rename_map
    )
)

for required_column in [
    "issuer_id",
    "security_id",
    "quantconnect_symbol",
    "quantconnect_market",
]:
    if (
        required_column
        not in quantconnect_security_map_df.columns
    ):
        quantconnect_security_map_df[
            required_column
        ] = pd.NA


# ------------------------------------------------------------
# 5. Compact QuantConnect long-format fundamentals
# ------------------------------------------------------------

quantconnect_source_columns = existing_columns(
    global_monthly_fundamental_panel_df,
    [
        "research_date",
        "issuer_id",
        "global_issuer_id",
        "security_id",
        "standard_concept",
        "effective_standard_concept",
        "reported_value",
        "reported_value_numeric",
        "effective_reported_value",
        "research_usable_value",
        "reported_currency",
        "currency",
        "period_start",
        "period_end",
        "period_type",
        "reporting_scope",
        "available_datetime",
        "filing_date",
        "fundamental_age_days",
        "is_stale",
        "source_region",
        "source_system",
        "source_quality_score",
        "research_usability",
        "block9_decision_applied",
        "block9_decision_direct",
        "block9_decision_propagated",
        "block9_issue_signature",
        "block9_mapping_basis",
        "block9_ai_confidence",
    ],
)

# Copy only the compact QuantConnect contract.
quantconnect_fundamental_long_df = (
    global_monthly_fundamental_panel_df.loc[
        :,
        quantconnect_source_columns,
    ]
    .reset_index(drop=True)
)

if (
    "research_date"
    in quantconnect_fundamental_long_df.columns
):
    quantconnect_fundamental_long_df[
        "date"
    ] = pd.to_datetime(
        quantconnect_fundamental_long_df[
            "research_date"
        ],
        errors="coerce",
    ).dt.normalize()

else:
    quantconnect_fundamental_long_df[
        "date"
    ] = pd.NaT


# ------------------------------------------------------------
# 6. Effective concept and value fallbacks
# ------------------------------------------------------------

if (
    "effective_standard_concept"
    not in quantconnect_fundamental_long_df.columns
):
    quantconnect_fundamental_long_df[
        "effective_standard_concept"
    ] = quantconnect_fundamental_long_df.get(
        "standard_concept",
        pd.Series(
            pd.NA,
            index=(
                quantconnect_fundamental_long_df
                .index
            ),
            dtype="string",
        ),
    )

if (
    "effective_reported_value"
    not in quantconnect_fundamental_long_df.columns
):
    value_source_column = first_existing_column(
        quantconnect_fundamental_long_df,
        [
            "research_usable_value",
            "reported_value_numeric",
            "reported_value",
        ],
    )

    if value_source_column is not None:
        quantconnect_fundamental_long_df[
            "effective_reported_value"
        ] = pd.to_numeric(
            quantconnect_fundamental_long_df[
                value_source_column
            ],
            errors="coerce",
        )

    else:
        quantconnect_fundamental_long_df[
            "effective_reported_value"
        ] = np.nan


# ------------------------------------------------------------
# 7. Attach QuantConnect symbols without multiplying rows
# ------------------------------------------------------------

security_join_columns = [
    column
    for column in [
        "issuer_id",
        "security_id",
    ]
    if (
        column
        in quantconnect_fundamental_long_df.columns
        and column
        in quantconnect_security_map_df.columns
    )
]

symbol_lookup_columns = (
    security_join_columns
    + [
        "quantconnect_symbol",
        "quantconnect_market",
    ]
)

symbol_lookup_df = (
    quantconnect_security_map_df.loc[
        :,
        list(
            dict.fromkeys(
                symbol_lookup_columns
            )
        ),
    ]
    .drop_duplicates(
        security_join_columns,
        keep="first",
    )
)

if security_join_columns:
    quantconnect_fundamental_long_df = (
        quantconnect_fundamental_long_df.merge(
            symbol_lookup_df,
            on=security_join_columns,
            how="left",
            validate="m:1",
        )
    )

else:
    quantconnect_fundamental_long_df[
        "quantconnect_symbol"
    ] = pd.NA

    quantconnect_fundamental_long_df[
        "quantconnect_market"
    ] = pd.NA

del symbol_lookup_df
release_unused_memory()


# ------------------------------------------------------------
# 8. Final QuantConnect column order
# ------------------------------------------------------------

quantconnect_final_columns = existing_columns(
    quantconnect_fundamental_long_df,
    [
        "date",
        "research_date",
        "issuer_id",
        "global_issuer_id",
        "security_id",
        "quantconnect_symbol",
        "quantconnect_market",
        "effective_standard_concept",
        "effective_reported_value",
        "standard_concept",
        "reported_value_numeric",
        "reported_currency",
        "currency",
        "period_start",
        "period_end",
        "period_type",
        "reporting_scope",
        "available_datetime",
        "filing_date",
        "fundamental_age_days",
        "is_stale",
        "source_region",
        "source_system",
        "source_quality_score",
        "research_usability",
        "block9_decision_applied",
        "block9_decision_direct",
        "block9_decision_propagated",
        "block9_issue_signature",
        "block9_mapping_basis",
        "block9_ai_confidence",
    ],
)

quantconnect_fundamental_long_df = (
    quantconnect_fundamental_long_df.loc[
        :,
        quantconnect_final_columns,
    ]
    .sort_values(
        existing_columns(
            quantconnect_fundamental_long_df,
            [
                "date",
                "quantconnect_symbol",
                "issuer_id",
                "effective_standard_concept",
            ],
        )
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Avoid duplicating the latent-factor matrix
# ------------------------------------------------------------

# This is intentionally an alias, not another complete copy.
quantconnect_monthly_feature_matrix_df = (
    latent_factor_model_matrix_df
)


# ------------------------------------------------------------
# 10. Validation
# ------------------------------------------------------------

duplicate_issuer_month_rows = 0

if {
    "research_date",
    "issuer_id",
}.issubset(
    latent_factor_model_matrix_df.columns
):
    duplicate_issuer_month_rows = int(
        latent_factor_model_matrix_df
        .duplicated(
            [
                "research_date",
                "issuer_id",
            ]
        )
        .sum()
    )


duplicate_quantconnect_rows = 0

quantconnect_duplicate_key = existing_columns(
    quantconnect_fundamental_long_df,
    [
        "date",
        "issuer_id",
        "security_id",
        "effective_standard_concept",
    ],
)

if quantconnect_duplicate_key:
    duplicate_quantconnect_rows = int(
        quantconnect_fundamental_long_df
        .duplicated(
            quantconnect_duplicate_key
        )
        .sum()
    )


lookahead_rows = 0

if {
    "date",
    "available_datetime",
}.issubset(
    quantconnect_fundamental_long_df.columns
):
    lookahead_rows = int(
        (
            pd.to_datetime(
                quantconnect_fundamental_long_df[
                    "available_datetime"
                ],
                errors="coerce",
                utc=True,
            )
            > pd.to_datetime(
                quantconnect_fundamental_long_df[
                    "date"
                ],
                errors="coerce",
                utc=True,
            )
        )
        .fillna(False)
        .sum()
    )


model_export_validation_df = pd.DataFrame({
    "metric": [
        "latent_factor_rows",
        "latent_factor_issuers",
        "latent_factor_months",
        "latent_factor_numeric_features",
        "duplicate_issuer_month_rows",
        "quantconnect_long_rows",
        "quantconnect_symbols",
        "duplicate_quantconnect_rows",
        "lookahead_rows",
        "block9_adjusted_long_rows",
        "block9_direct_long_rows",
        "block9_propagated_long_rows",
    ],
    "value": [
        len(
            latent_factor_model_matrix_df
        ),
        safe_nunique(
            latent_factor_model_matrix_df,
            "issuer_id",
        ),
        safe_nunique(
            latent_factor_model_matrix_df,
            "research_date",
        ),
        len(
            latent_numeric_feature_columns
        ),
        duplicate_issuer_month_rows,
        len(
            quantconnect_fundamental_long_df
        ),
        safe_nunique(
            quantconnect_fundamental_long_df,
            "quantconnect_symbol",
        ),
        duplicate_quantconnect_rows,
        lookahead_rows,
        int(
            quantconnect_fundamental_long_df.get(
                "block9_decision_applied",
                pd.Series(
                    False,
                    index=(
                        quantconnect_fundamental_long_df
                        .index
                    ),
                ),
            )
            .fillna(False)
            .sum()
        ),
        int(
            quantconnect_fundamental_long_df.get(
                "block9_decision_direct",
                pd.Series(
                    False,
                    index=(
                        quantconnect_fundamental_long_df
                        .index
                    ),
                ),
            )
            .fillna(False)
            .sum()
        ),
        int(
            quantconnect_fundamental_long_df.get(
                "block9_decision_propagated",
                pd.Series(
                    False,
                    index=(
                        quantconnect_fundamental_long_df
                        .index
                    ),
                ),
            )
            .fillna(False)
            .sum()
        ),
    ],
})


print(
    "Latent-factor model rows:",
    f"{len(latent_factor_model_matrix_df):,}",
)

print(
    "Latent-factor numeric features:",
    f"{len(latent_numeric_feature_columns):,}",
)

print(
    "QuantConnect long-format rows:",
    f"{len(quantconnect_fundamental_long_df):,}",
)

display(
    model_export_validation_df
)

release_unused_memory()


Latent-factor model rows: 10,711
Latent-factor numeric features: 21
QuantConnect long-format rows: 3,180,784


,metric,value
0,latent_factor_rows,10711
1,latent_factor_issuers,168
2,latent_factor_months,81
3,latent_factor_numeric_features,21
4,duplicate_issuer_month_rows,0
5,quantconnect_long_rows,3180784
6,quantconnect_symbols,103
7,duplicate_quantconnect_rows,3063431
8,lookahead_rows,0
9,block9_adjusted_long_rows,0


In [ ]:
# 19. BLOCK 10 OUTPUT CONTRACT

block_10_data = {
    "block_9_quality_control_outcomes_df": block_9_quality_control_outcomes_df,
    "block_9_quality_control_summary_df": block_9_quality_control_summary_df,
    "block_9_overlay_audit_df": block_9_overlay_audit_df,
    "global_fundamentals_pre_block9_df": global_fundamentals_pre_block9_df,
    "latent_factor_model_matrix_df": latent_factor_model_matrix_df,
    "latent_factor_feature_dictionary_df": latent_factor_feature_dictionary_df,
    "quantconnect_fundamental_long_df": quantconnect_fundamental_long_df,
    "quantconnect_security_map_df": quantconnect_security_map_df,
    "quantconnect_monthly_feature_matrix_df": quantconnect_monthly_feature_matrix_df,
    "model_export_validation_df": model_export_validation_df,
    # Load and architecture logs
    "regional_manifest_status_df": regional_manifest_status_df,
    "regional_architecture_load_log_df": regional_architecture_load_log_df,
    "regional_concept_load_log_df": regional_concept_load_log_df,
    "regional_fact_load_log_df": regional_fact_load_log_df,
    "global_filing_load_log_df": global_filing_load_log_df,

    # Issuer-centric global architecture
    "global_economic_issuer_universe_df": global_economic_issuer_universe_df,
    "global_issuer_security_universe_df": global_issuer_security_universe_df,
    "global_entity_relationship_graph_df": global_entity_relationship_graph_df,

    # Global concept registry
    "regional_concept_dictionary_df": regional_concept_dictionary_df,
    "global_standard_concept_dictionary_df": global_standard_concept_dictionary_df,
    "global_concept_definition_conflicts_df": global_concept_definition_conflicts_df,
    "global_regional_concept_coverage_df": global_regional_concept_coverage_df,

    # Preferred-source architecture
    "global_source_precedence_df": global_source_precedence_df,
    "global_preferred_accounting_source_df": global_preferred_accounting_source_df,

    # Filing and fact stores
    "global_filing_metadata_df": global_filing_metadata_df,
    "global_fundamentals_combined_df": global_fundamentals_combined_df,
    "global_fundamentals_selected_df": global_fundamentals_selected_df,
    "global_fundamentals_alternatives_df": global_fundamentals_alternatives_df,
    "global_incremental_facts_df": global_incremental_facts_df,
    "global_incremental_facts_selected_df": global_incremental_facts_selected_df,
    "global_duplicate_resolution_df": global_duplicate_resolution_df,
    "global_fact_version_history_df": global_fact_version_history_df,

    # Point-in-time research data
    "global_monthly_fundamental_panel_df": global_monthly_fundamental_panel_df,
    "global_monthly_accounting_wide_df": global_monthly_accounting_wide_df,
    "global_monthly_interpretation_features_df": global_monthly_interpretation_features_df,

    # Coverage and quality
    "global_region_coverage_df": global_region_coverage_df,
    "global_concept_coverage_df": global_concept_coverage_df,
    "global_issuer_coverage_df": global_issuer_coverage_df,
    "global_feature_coverage_df": global_feature_coverage_df,
    "global_issuer_feature_coverage_df": global_issuer_feature_coverage_df,
    "global_monthly_feature_coverage_df": global_monthly_feature_coverage_df,
    "global_quality_report_df": global_quality_report_df,
    "global_research_usability_report_df": global_research_usability_report_df,
}

print("Block 10 transformations complete.")

for name in [
    "global_fundamentals_selected_df",
    "global_incremental_facts_selected_df",
    "global_monthly_fundamental_panel_df",
    "global_monthly_interpretation_features_df",
]:
    print(
        f"  {name}: "
        f"{len(block_10_data[name]):,} rows"
    )


Block 10 transformations complete.
  global_fundamentals_selected_df: 459,215 rows
  global_incremental_facts_selected_df: 0 rows
  global_monthly_fundamental_panel_df: 3,180,784 rows
  global_monthly_interpretation_features_df: 10,711 rows


In [ ]:
# 20. MEMORY-SAFE PERSISTENCE

def release_unused_memory():
    gc.collect()

    try:
        ctypes.CDLL(
            "libc.so.6"
        ).malloc_trim(0)
    except Exception:
        pass


def infer_stable_arrow_schema(
    dataframe,
):
    template = dataframe.head(
        0
    ).copy()

    for column in template.columns:
        dtype = dataframe[column].dtype

        if (
            pd.api.types.is_object_dtype(
                dtype
            )
            or pd.api.types.is_string_dtype(
                dtype
            )
            or isinstance(
                dtype,
                pd.CategoricalDtype,
            )
        ):
            template[column] = (
                pd.Series(
                    dtype="string"
                )
            )

        elif pd.api.types.is_bool_dtype(
            dtype
        ):
            template[column] = (
                pd.Series(
                    dtype="boolean"
                )
            )

    schema = pa.Schema.from_pandas(
        template,
        preserve_index=False,
    )

    del template
    release_unused_memory()

    return schema


def prepare_chunk_for_schema(
    chunk,
    arrow_schema,
):
    output = chunk.copy(
        deep=False
    )

    for field in arrow_schema:
        column = field.name

        if column not in output.columns:
            continue

        series = output[column]

        if pa.types.is_string(
            field.type
        ):
            output[column] = (
                series.astype(
                    "string"
                )
            )

        elif pa.types.is_timestamp(
            field.type
        ):
            converted = pd.to_datetime(
                series,
                errors="coerce",
                utc=(
                    field.type.tz
                    is not None
                ),
            )

            if field.type.tz is None:
                try:
                    converted = (
                        converted
                        .dt.tz_localize(
                            None
                        )
                    )
                except Exception:
                    pass

            output[column] = (
                converted
            )

        elif pa.types.is_boolean(
            field.type
        ):
            output[column] = (
                series.astype(
                    "boolean"
                )
            )

        elif pa.types.is_integer(
            field.type
        ):
            output[column] = (
                pd.to_numeric(
                    series,
                    errors="coerce",
                )
                .round()
                .astype("Int64")
            )

        elif pa.types.is_floating(
            field.type
        ):
            output[column] = (
                pd.to_numeric(
                    series,
                    errors="coerce",
                )
            )

    return output


def persist_dataframe_streaming(
    name,
    dataframe,
    output_dir,
    *,
    chunk_rows=PARQUET_CHUNK_ROWS,
    overwrite=True,
):
    output_path = (
        output_dir
        / f"{name}.parquet"
    )

    if output_path.exists():
        if overwrite:
            output_path.unlink()
        else:
            raise FileExistsError(
                output_path
            )

    row_count = len(
        dataframe
    )
    column_count = len(
        dataframe.columns
    )

    print(
        f"Persisting {name}: "
        f"{row_count:,} rows × "
        f"{column_count:,} columns"
    )

    arrow_schema = (
        infer_stable_arrow_schema(
            dataframe
        )
    )

    writer = None

    try:
        writer = pq.ParquetWriter(
            output_path,
            arrow_schema,
            compression="snappy",
            use_dictionary=True,
            write_statistics=True,
        )

        if row_count == 0:
            empty = dataframe.head(
                0
            ).copy()

            table = pa.Table.from_pandas(
                empty,
                schema=arrow_schema,
                preserve_index=False,
                safe=False,
            )

            writer.write_table(
                table
            )

        else:
            for start in range(
                0,
                row_count,
                chunk_rows,
            ):
                end = min(
                    start
                    + chunk_rows,
                    row_count,
                )

                chunk = dataframe.iloc[
                    start:end
                ]

                prepared = (
                    prepare_chunk_for_schema(
                        chunk,
                        arrow_schema,
                    )
                )

                table = pa.Table.from_pandas(
                    prepared,
                    schema=arrow_schema,
                    preserve_index=False,
                    safe=False,
                )

                writer.write_table(
                    table,
                    row_group_size=(
                        len(table)
                    ),
                )

                del table
                del prepared
                del chunk

                release_unused_memory()

    except Exception:
        if writer is not None:
            writer.close()
            writer = None

        if output_path.exists():
            output_path.unlink()

        release_unused_memory()
        raise

    finally:
        if writer is not None:
            writer.close()

    return {
        "table_name": name,
        "path": str(
            output_path
        ),
        "row_count": int(
            row_count
        ),
        "column_count": int(
            column_count
        ),
        "columns": list(
            map(
                str,
                dataframe.columns,
            )
        ),
        "file_size_bytes": int(
            output_path.stat().st_size
        ),
        "created_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
    }


if PERSIST_BLOCK_10_OUTPUTS:
    release_unused_memory()

    manifest_rows = []

    preferred_order = [
        "global_economic_issuer_universe_df",
        "global_issuer_security_universe_df",
        "global_standard_concept_dictionary_df",
        "global_preferred_accounting_source_df",
        "global_fundamentals_selected_df",
        "global_incremental_facts_selected_df",
        "global_fact_version_history_df",
        "global_monthly_interpretation_features_df",
        "global_feature_coverage_df",
        "global_quality_report_df",
        "global_research_usability_report_df",
        "global_filing_metadata_df",
        "global_fundamentals_alternatives_df",
        "global_duplicate_resolution_df",
        "global_fundamentals_combined_df",
        "global_monthly_fundamental_panel_df",
    ]

    persistence_order = (
        [
            name
            for name in preferred_order
            if name in block_10_data
        ]
        + [
            name
            for name in block_10_data
            if name not in preferred_order
        ]
    )

    for table_name in persistence_order:
        record = persist_dataframe_streaming(
            table_name,
            block_10_data[
                table_name
            ],
            BLOCK_10_OUTPUT_DIR,
            chunk_rows=(
                PARQUET_CHUNK_ROWS
            ),
            overwrite=(
                OVERWRITE_PERSISTED_OUTPUTS
            ),
        )

        manifest_rows.append(
            record
        )

        release_unused_memory()

    block_10_manifest = {
        "block": 10,
        "block_name": (
            "Global point-in-time fundamentals "
            "and interpretation features"
        ),
        "created_at_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "project_root": str(
            PROJECT_ROOT
        ),
        "input_manifests": {
            region: str(path)
            for region, path
            in REGIONAL_MANIFESTS.items()
        },
        "output_directory": str(
            BLOCK_10_OUTPUT_DIR
        ),
        "canonical_concept_count": int(
            global_standard_concept_dictionary_df[
                "standard_concept"
            ].nunique()
        ),
        "preferred_source_description": (
            PREFERRED_SOURCE_DESCRIPTION
        ),
        "monthly_panel_start": str(
            PANEL_START_DATE.date()
        ),
        "monthly_panel_end": str(
            PANEL_END_DATE.date()
        ),
        "parquet_chunk_rows": (
            PARQUET_CHUNK_ROWS
        ),
        "tables": manifest_rows,
    }

    with BLOCK_10_MANIFEST_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            block_10_manifest,
            file,
            indent=2,
            ensure_ascii=False,
        )

    block_10_persistence_report_df = (
        pd.DataFrame(
            manifest_rows
        )
    )

    print(
        "Block 10 outputs persisted successfully."
    )
    print(
        "Manifest:",
        BLOCK_10_MANIFEST_PATH,
    )

    display(
        block_10_persistence_report_df[
            [
                "table_name",
                "row_count",
                "column_count",
                "file_size_bytes",
                "path",
            ]
        ]
    )


Persisting global_economic_issuer_universe_df: 326 rows × 6 columns
Persisting global_issuer_security_universe_df: 318 rows × 10 columns
Persisting global_standard_concept_dictionary_df: 132 rows × 9 columns
Persisting global_preferred_accounting_source_df: 169 rows × 11 columns
Persisting global_fundamentals_selected_df: 459,215 rows × 102 columns
Persisting global_incremental_facts_selected_df: 0 rows × 58 columns
Persisting global_fact_version_history_df: 459,215 rows × 19 columns
Persisting global_monthly_interpretation_features_df: 10,711 rows × 26 columns
Persisting global_feature_coverage_df: 21 rows × 7 columns
Persisting global_quality_report_df: 12 rows × 2 columns
Persisting global_research_usability_report_df: 15 rows × 6 columns
Persisting global_filing_metadata_df: 21,549 rows × 21 columns
Persisting global_fundamentals_alternatives_df: 18,159 rows × 87 columns
Persisting global_duplicate_resolution_df: 477,374 rows × 95 columns
Persisting global_fundamentals_combined_df:

KeyboardInterrupt: 

In [ ]:
# 21. PERSISTENCE VALIDATION

if PERSIST_BLOCK_10_OUTPUTS:
    required_tables = {
        "global_economic_issuer_universe_df",
        "global_issuer_security_universe_df",
        "global_standard_concept_dictionary_df",
        "global_preferred_accounting_source_df",
        "global_filing_metadata_df",
        "global_fundamentals_combined_df",
        "global_fundamentals_selected_df",
        "global_fundamentals_alternatives_df",
        "global_duplicate_resolution_df",
        "global_fact_version_history_df",
        "global_monthly_fundamental_panel_df",
        "global_monthly_interpretation_features_df",
        "global_feature_coverage_df",
        "global_quality_report_df",
        "global_research_usability_report_df",
        "block_9_quality_control_summary_df",
        "block_9_overlay_audit_df",
        "latent_factor_model_matrix_df",
        "quantconnect_fundamental_long_df",
        "quantconnect_security_map_df",
        "model_export_validation_df",
    }

    manifest_names = {
        item["table_name"]
        for item in block_10_manifest[
            "tables"
        ]
    }

    missing = (
        required_tables
        .difference(
            manifest_names
        )
    )

    if missing:
        raise RuntimeError(
            "Persistence validation "
            f"missing tables: {sorted(missing)}"
        )

    validation_rows = []

    for table_name in sorted(
        required_tables
    ):
        table_path = (
            BLOCK_10_OUTPUT_DIR
            / f"{table_name}.parquet"
        )

        if not table_path.exists():
            raise FileNotFoundError(
                table_path
            )

        parquet_file = (
            pq.ParquetFile(
                table_path
            )
        )

        original_rows = len(
            block_10_data[
                table_name
            ]
        )

        persisted_rows = (
            parquet_file
            .metadata
            .num_rows
        )

        persisted_columns = (
            parquet_file
            .metadata
            .num_columns
        )

        if (
            original_rows
            != persisted_rows
        ):
            raise RuntimeError(
                f"Row-count mismatch for "
                f"{table_name}: "
                f"{original_rows:,} original "
                f"versus {persisted_rows:,} persisted."
            )

        validation_rows.append({
            "table_name": table_name,
            "original_rows": (
                original_rows
            ),
            "persisted_rows": (
                persisted_rows
            ),
            "persisted_columns": (
                persisted_columns
            ),
            "status": "PASSED",
        })

        del parquet_file
        release_unused_memory()

    block_10_validation_report_df = (
        pd.DataFrame(
            validation_rows
        )
    )

    display(
        block_10_validation_report_df
    )

    print(
        "Block 10 persistence validation passed. "
        "The latent-factor discovery and supervised "
        "interpretation modules can load the versioned "
        "global research dataset without rerunning regional "
        "filing collection."
    )

In [ ]:
# 22. FINAL DIAGNOSTIC — BLOCK 10 GLOBAL PIPELINE

print("=" * 96)
print("BLOCK 10 — FINAL GLOBAL PIPELINE DIAGNOSTIC")
print("=" * 96)

# ------------------------------------------------------------
# 1. Safe helpers
# ------------------------------------------------------------

def get_frame(variable_name):
    value = globals().get(variable_name)

    if isinstance(value, pd.DataFrame):
        return value

    return pd.DataFrame()


def safe_unique(frame, column):
    if frame.empty or column not in frame.columns:
        return 0

    return int(
        frame[column].nunique(
            dropna=True
        )
    )


def safe_non_null(frame, column):
    if frame.empty or column not in frame.columns:
        return 0

    return int(
        frame[column].notna().sum()
    )


def safe_bool_sum(frame, column):
    if frame.empty or column not in frame.columns:
        return 0

    return int(
        frame[column]
        .fillna(False)
        .astype(bool)
        .sum()
    )


def metric_value(frame, metric_name, default=np.nan):
    if (
        frame.empty
        or "metric" not in frame.columns
        or "value" not in frame.columns
    ):
        return default

    matched = frame.loc[
        frame["metric"]
        .astype("string")
        .eq(metric_name),
        "value",
    ]

    if matched.empty:
        return default

    return matched.iloc[0]


# ------------------------------------------------------------
# 2. Load principal Block 10 tables
# ------------------------------------------------------------

combined_df = get_frame(
    "global_fundamentals_combined_df"
)

selected_df = get_frame(
    "global_fundamentals_selected_df"
)

alternatives_df = get_frame(
    "global_fundamentals_alternatives_df"
)

duplicate_resolution_df = get_frame(
    "global_duplicate_resolution_df"
)

version_history_df = get_frame(
    "global_fact_version_history_df"
)

filing_metadata_df = get_frame(
    "global_filing_metadata_df"
)

issuer_security_df = get_frame(
    "global_issuer_security_universe_df"
)

economic_issuer_df = get_frame(
    "global_economic_issuer_universe_df"
)

preferred_source_df = get_frame(
    "global_preferred_accounting_source_df"
)

monthly_panel_df = get_frame(
    "global_monthly_fundamental_panel_df"
)

interpretation_df = get_frame(
    "global_monthly_interpretation_features_df"
)

latent_matrix_df = get_frame(
    "latent_factor_model_matrix_df"
)

qc_long_df = get_frame(
    "quantconnect_fundamental_long_df"
)

qc_security_map_df = get_frame(
    "quantconnect_security_map_df"
)

feature_coverage_df = get_frame(
    "global_feature_coverage_df"
)

quality_report_df = get_frame(
    "global_quality_report_df"
)

research_usability_df = get_frame(
    "global_research_usability_report_df"
)

model_export_validation_df = get_frame(
    "model_export_validation_df"
)

block9_summary_df = get_frame(
    "block_9_quality_control_summary_df"
)

block9_region_df = get_frame(
    "block_9_feedback_by_region_df"
)

persistence_validation_df = get_frame(
    "block_10_validation_report_df"
)


# ------------------------------------------------------------
# 3. Global row reconciliation
# ------------------------------------------------------------

combined_rows = len(combined_df)
selected_rows = len(selected_df)
alternative_rows = len(alternatives_df)

row_reconciliation_difference = (
    combined_rows
    - selected_rows
    - alternative_rows
)

row_reconciliation_df = pd.DataFrame({
    "metric": [
        "combined_rows",
        "selected_rows",
        "alternative_rows",
        "selected_plus_alternatives",
        "reconciliation_difference",
    ],
    "value": [
        combined_rows,
        selected_rows,
        alternative_rows,
        selected_rows + alternative_rows,
        row_reconciliation_difference,
    ],
})


print("\n1. GLOBAL ROW RECONCILIATION")
display(
    row_reconciliation_df
)


# ------------------------------------------------------------
# 4. Regional coverage
# ------------------------------------------------------------

if (
    not combined_df.empty
    and "source_region" in combined_df.columns
):
    regional_coverage_df = (
        combined_df
        .groupby(
            "source_region",
            dropna=False,
        )
        .agg(
            combined_rows=(
                "standard_concept",
                "size",
            ),
            unique_issuers=(
                "issuer_id",
                "nunique",
            ),
            unique_securities=(
                "security_id",
                "nunique",
            ),
            unique_documents=(
                "document_id",
                "nunique",
            ),
            unique_concepts=(
                "standard_concept",
                "nunique",
            ),
        )
        .reset_index()
        .sort_values(
            "combined_rows",
            ascending=False,
        )
    )

else:
    regional_coverage_df = pd.DataFrame()


print("\n2. REGIONAL COVERAGE")
display(
    regional_coverage_df
)


# ------------------------------------------------------------
# 5. Global identifier integrity
# ------------------------------------------------------------

identifier_columns = [
    "global_source_observation_id",
    "issuer_id",
    "security_id",
    "document_id",
    "filing_id",
    "standard_concept",
    "available_datetime",
]

identifier_rows = []

for column in identifier_columns:
    duplicate_rows = 0

    if (
        column
        in combined_df.columns
        and column
        == "global_source_observation_id"
    ):
        duplicate_rows = int(
            combined_df[column]
            .duplicated(
                keep=False
            )
            .sum()
        )

    identifier_rows.append({
        "column": column,
        "present": (
            column in combined_df.columns
        ),
        "non_null_rows": safe_non_null(
            combined_df,
            column,
        ),
        "unique_values": safe_unique(
            combined_df,
            column,
        ),
        "duplicate_rows": duplicate_rows,
    })


global_identifier_integrity_df = pd.DataFrame(
    identifier_rows
)


print("\n3. GLOBAL IDENTIFIER INTEGRITY")
display(
    global_identifier_integrity_df
)


# ------------------------------------------------------------
# 6. Block 9 upstream feedback validation
# ------------------------------------------------------------

print("\n4. BLOCK 9 UPSTREAM FEEDBACK")

if block9_region_df.empty:
    print(
        "block_9_feedback_by_region_df is unavailable."
    )
else:
    display(
        block9_region_df
    )

if block9_summary_df.empty:
    print(
        "block_9_quality_control_summary_df is unavailable."
    )
else:
    display(
        block9_summary_df
    )


block9_contracts_expected = metric_value(
    block9_summary_df,
    "regional_feedback_contracts_expected",
    default=0,
)

block9_contracts_passed = metric_value(
    block9_summary_df,
    "regional_feedback_contracts_passed",
    default=0,
)

block9_retro_updates = metric_value(
    block9_summary_df,
    "global_fact_rows_updated_in_block10",
    default=np.nan,
)


# ------------------------------------------------------------
# 7. Selection and versioning integrity
# ------------------------------------------------------------

selection_summary_df = pd.DataFrame({
    "metric": [
        "duplicate_resolution_rows",
        "selected_fact_rows",
        "alternative_fact_rows",
        "fact_version_history_rows",
        "preferred_source_rows",
        "filing_metadata_rows",
        "issuer_security_rows",
        "economic_issuer_rows",
    ],
    "value": [
        len(duplicate_resolution_df),
        len(selected_df),
        len(alternatives_df),
        len(version_history_df),
        len(preferred_source_df),
        len(filing_metadata_df),
        len(issuer_security_df),
        len(economic_issuer_df),
    ],
})


print("\n5. SELECTION AND VERSIONING")
display(
    selection_summary_df
)


# ------------------------------------------------------------
# 8. Monthly panel and model-export integrity
# ------------------------------------------------------------

model_pipeline_summary_df = pd.DataFrame({
    "table": [
        "global_monthly_fundamental_panel_df",
        "global_monthly_interpretation_features_df",
        "latent_factor_model_matrix_df",
        "quantconnect_fundamental_long_df",
        "quantconnect_security_map_df",
        "global_feature_coverage_df",
    ],
    "rows": [
        len(monthly_panel_df),
        len(interpretation_df),
        len(latent_matrix_df),
        len(qc_long_df),
        len(qc_security_map_df),
        len(feature_coverage_df),
    ],
    "columns": [
        len(monthly_panel_df.columns),
        len(interpretation_df.columns),
        len(latent_matrix_df.columns),
        len(qc_long_df.columns),
        len(qc_security_map_df.columns),
        len(feature_coverage_df.columns),
    ],
})


print("\n6. MONTHLY PANEL AND MODEL EXPORTS")
display(
    model_pipeline_summary_df
)


# ------------------------------------------------------------
# 9. Latent-factor matrix quality
# ------------------------------------------------------------

latent_numeric_columns = (
    latent_matrix_df
    .select_dtypes(
        include=[
            "number",
            "bool",
        ]
    )
    .columns
    .tolist()
    if not latent_matrix_df.empty
    else []
)

latent_missing_cells = (
    int(
        latent_matrix_df[
            latent_numeric_columns
        ]
        .isna()
        .sum()
        .sum()
    )
    if latent_numeric_columns
    else 0
)

latent_total_numeric_cells = (
    int(
        len(latent_matrix_df)
        * len(latent_numeric_columns)
    )
    if latent_numeric_columns
    else 0
)

latent_missing_ratio = (
    latent_missing_cells
    / latent_total_numeric_cells
    if latent_total_numeric_cells > 0
    else np.nan
)


latent_quality_df = pd.DataFrame({
    "metric": [
        "latent_rows",
        "latent_columns",
        "numeric_feature_columns",
        "missing_numeric_cells",
        "total_numeric_cells",
        "missing_numeric_ratio",
        "duplicate_rows",
    ],
    "value": [
        len(latent_matrix_df),
        len(latent_matrix_df.columns),
        len(latent_numeric_columns),
        latent_missing_cells,
        latent_total_numeric_cells,
        latent_missing_ratio,
        int(
            latent_matrix_df
            .duplicated()
            .sum()
        )
        if not latent_matrix_df.empty
        else 0,
    ],
})


print("\n7. LATENT-FACTOR MATRIX QUALITY")
display(
    latent_quality_df
)


# ------------------------------------------------------------
# 10. QuantConnect export reconciliation
# ------------------------------------------------------------

quantconnect_reconciliation_df = pd.DataFrame({
    "metric": [
        "monthly_panel_rows",
        "quantconnect_long_rows",
        "row_difference",
        "security_map_rows",
        "issuer_security_universe_rows",
        "security_map_difference",
    ],
    "value": [
        len(monthly_panel_df),
        len(qc_long_df),
        len(qc_long_df) - len(monthly_panel_df),
        len(qc_security_map_df),
        len(issuer_security_df),
        len(qc_security_map_df)
        - len(issuer_security_df),
    ],
})


print("\n8. QUANTCONNECT EXPORT RECONCILIATION")
display(
    quantconnect_reconciliation_df
)


# ------------------------------------------------------------
# 11. Persistence validation
# ------------------------------------------------------------

if persistence_validation_df.empty:
    persistence_summary_df = pd.DataFrame(
        columns=[
            "metric",
            "value",
        ]
    )

    persistence_all_passed = False

else:
    persistence_passed_rows = int(
        persistence_validation_df[
            "status"
        ]
        .astype("string")
        .eq("PASSED")
        .sum()
    )

    persistence_all_passed = (
        persistence_passed_rows
        == len(
            persistence_validation_df
        )
    )

    persisted_row_mismatches = int(
        (
            pd.to_numeric(
                persistence_validation_df[
                    "original_rows"
                ],
                errors="coerce",
            )
            != pd.to_numeric(
                persistence_validation_df[
                    "persisted_rows"
                ],
                errors="coerce",
            )
        )
        .fillna(True)
        .sum()
    )

    persistence_summary_df = pd.DataFrame({
        "metric": [
            "persisted_tables",
            "passed_tables",
            "failed_tables",
            "row_count_mismatches",
        ],
        "value": [
            len(
                persistence_validation_df
            ),
            persistence_passed_rows,
            (
                len(
                    persistence_validation_df
                )
                - persistence_passed_rows
            ),
            persisted_row_mismatches,
        ],
    })


print("\n9. PERSISTENCE VALIDATION")
display(
    persistence_summary_df
)


# ------------------------------------------------------------
# 12. Final pass/fail gates
# ------------------------------------------------------------

global_id_unique = (
    "global_source_observation_id"
    in combined_df.columns
    and safe_non_null(
        combined_df,
        "global_source_observation_id",
    )
    == len(combined_df)
    and safe_unique(
        combined_df,
        "global_source_observation_id",
    )
    == len(combined_df)
)

regional_coverage_present = (
    not regional_coverage_df.empty
    and regional_coverage_df[
        "source_region"
    ].nunique(
        dropna=True
    )
    >= 7
)

row_reconciliation_passed = (
    row_reconciliation_difference == 0
)

block9_feedback_passed = (
    pd.to_numeric(
        pd.Series([
            block9_contracts_passed
        ]),
        errors="coerce",
    )
    .fillna(0)
    .iloc[0]
    ==
    pd.to_numeric(
        pd.Series([
            block9_contracts_expected
        ]),
        errors="coerce",
    )
    .fillna(-1)
    .iloc[0]
    and pd.to_numeric(
        pd.Series([
            block9_contracts_expected
        ]),
        errors="coerce",
    )
    .fillna(0)
    .iloc[0]
    > 0
)

no_retroactive_block10_updates = (
    pd.to_numeric(
        pd.Series([
            block9_retro_updates
        ]),
        errors="coerce",
    )
    .fillna(0)
    .iloc[0]
    == 0
)

monthly_panel_non_empty = (
    len(monthly_panel_df) > 0
)

latent_matrix_non_empty = (
    len(latent_matrix_df) > 0
)

quantconnect_rows_match = (
    len(monthly_panel_df)
    == len(qc_long_df)
)

quantconnect_security_map_matches = (
    len(qc_security_map_df)
    == len(issuer_security_df)
)

latent_duplicates_zero = (
    latent_matrix_df
    .duplicated()
    .sum()
    == 0
    if not latent_matrix_df.empty
    else False
)


final_checks = [
    {
        "check":
            "Combined rows reconcile to selected plus alternatives",
        "passed":
            row_reconciliation_passed,
        "detail":
            row_reconciliation_difference,
    },
    {
        "check":
            "Global observation IDs are complete and unique",
        "passed":
            global_id_unique,
        "detail":
            len(combined_df),
    },
    {
        "check":
            "All seven regional sources are represented",
        "passed":
            regional_coverage_present,
        "detail":
            safe_unique(
                combined_df,
                "source_region",
            ),
    },
    {
        "check":
            "All Block 9 regional feedback contracts passed",
        "passed":
            block9_feedback_passed,
        "detail":
            (
                f"{block9_contracts_passed}/"
                f"{block9_contracts_expected}"
            ),
    },
    {
        "check":
            "Block 10 performed no retrospective AI overlay",
        "passed":
            no_retroactive_block10_updates,
        "detail":
            block9_retro_updates,
    },
    {
        "check":
            "Monthly fundamental panel is non-empty",
        "passed":
            monthly_panel_non_empty,
        "detail":
            len(monthly_panel_df),
    },
    {
        "check":
            "Latent-factor matrix is non-empty",
        "passed":
            latent_matrix_non_empty,
        "detail":
            len(latent_matrix_df),
    },
    {
        "check":
            "Latent-factor matrix has no duplicate rows",
        "passed":
            latent_duplicates_zero,
        "detail":
            (
                int(
                    latent_matrix_df
                    .duplicated()
                    .sum()
                )
                if not latent_matrix_df.empty
                else np.nan
            ),
    },
    {
        "check":
            "QuantConnect long export matches monthly panel rows",
        "passed":
            quantconnect_rows_match,
        "detail":
            (
                f"{len(qc_long_df):,} / "
                f"{len(monthly_panel_df):,}"
            ),
    },
    {
        "check":
            "QuantConnect security map matches security universe",
        "passed":
            quantconnect_security_map_matches,
        "detail":
            (
                f"{len(qc_security_map_df):,} / "
                f"{len(issuer_security_df):,}"
            ),
    },
    {
        "check":
            "All persisted tables passed validation",
        "passed":
            persistence_all_passed,
        "detail":
            len(
                persistence_validation_df
            ),
    },
]


block_10_final_pipeline_checks_df = pd.DataFrame(
    final_checks
)


print("\n10. FINAL PIPELINE CHECKS")
display(
    block_10_final_pipeline_checks_df
)


failed_checks_df = (
    block_10_final_pipeline_checks_df.loc[
        ~block_10_final_pipeline_checks_df[
            "passed"
        ]
        .fillna(False)
        .astype(bool)
    ]
    .copy()
)


if failed_checks_df.empty:
    print(
        "\nBLOCK 10 GLOBAL PIPELINE DIAGNOSTIC: PASSED"
    )

    print(
        "The versioned global dataset, monthly panel, "
        "latent-factor matrix and QuantConnect exports are "
        "ready for downstream modelling."
    )

else:
    print(
        "\nBLOCK 10 GLOBAL PIPELINE DIAGNOSTIC: "
        f"{len(failed_checks_df)} CHECK(S) FAILED"
    )

    display(
        failed_checks_df
    )


print("=" * 96)


In [ ]:
# ============================================================
# 23. EXPORT GLOBAL DATASET AS REGIONAL EXCEL WORKBOOKS
# ============================================================

# Output folder:
#
# data/exports/global_dataset_excel/
#
# This is intentionally outside:
# data/interim/block_10/


# ------------------------------------------------------------
# 1. Ensure XlsxWriter is available
# ------------------------------------------------------------

if importlib.util.find_spec("xlsxwriter") is None:
    print("Installing xlsxwriter...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "xlsxwriter",
    ])

import xlsxwriter


# ------------------------------------------------------------
# 2. Export configuration
# ------------------------------------------------------------

EXPORT_GLOBAL_DATASET_TO_EXCEL = True

GLOBAL_EXCEL_EXPORT_DIR = (
    DATA_ROOT
    / "exports"
    / "global_dataset_excel"
)

GLOBAL_EXCEL_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SUMMARY_WORKBOOK_PATH = (
    GLOBAL_EXCEL_EXPORT_DIR
    / "Global_Automotive_Research_Dataset_Summary.xlsx"
)


REGION_FILE_NAMES = {
    "USA":
        "Global_Automotive_Dataset_USA.xlsx",

    "EUROPE":
        "Global_Automotive_Dataset_Europe.xlsx",

    "JAPAN":
        "Global_Automotive_Dataset_Japan.xlsx",

    "KOREA":
        "Global_Automotive_Dataset_Korea.xlsx",

    "MAINLAND_CHINA":
        "Global_Automotive_Dataset_Mainland_China.xlsx",

    "HONG_KONG":
        "Global_Automotive_Dataset_Hong_Kong.xlsx",

    "AUSTRALIA":
        "Global_Automotive_Dataset_Australia.xlsx",
}


# ------------------------------------------------------------
# 3. Excel-safe value conversion
# ------------------------------------------------------------

ILLEGAL_XML_CHARACTERS = re.compile(
    r"[\x00-\x08\x0B\x0C\x0E-\x1F]"
)


def excel_safe_value(value):
    """
    Convert individual values into forms that XlsxWriter and
    Excel can represent safely.
    """
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, pd.Timestamp):
        if value.tzinfo is not None:
            value = (
                value
                .tz_convert("UTC")
                .tz_localize(None)
            )

        return value.to_pydatetime()

    if isinstance(value, np.datetime64):
        converted = pd.Timestamp(value)

        if pd.isna(converted):
            return None

        return converted.to_pydatetime()

    if isinstance(value, datetime):
        if value.tzinfo is not None:
            value = (
                value
                .astimezone(timezone.utc)
                .replace(tzinfo=None)
            )

        return value

    if isinstance(value, date):
        return value

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, float):
        if not np.isfinite(value):
            return None

        return value

    if isinstance(value, str):
        return ILLEGAL_XML_CHARACTERS.sub(
            "",
            value,
        )

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
            dict,
        ),
    ):
        return ILLEGAL_XML_CHARACTERS.sub(
            "",
            str(value),
        )

    return value


# ------------------------------------------------------------
# 4. Stream a DataFrame to Excel row by row
# ------------------------------------------------------------

def write_dataframe_streaming(
    dataframe,
    output_path,
    worksheet_name="Global_Dataset",
    title=None,
):
    """
    Write a large DataFrame using XlsxWriter's genuine
    row-streaming constant-memory mode.

    This avoids pandas.to_excel(), which is incompatible with
    XlsxWriter constant_memory mode.
    """
    output_path = Path(
        output_path
    )

    if output_path.exists():
        output_path.unlink()

    workbook = xlsxwriter.Workbook(
        str(output_path),
        {
            "constant_memory": True,
            "strings_to_urls": False,
            "nan_inf_to_errors": False,
        },
    )

    worksheet = workbook.add_worksheet(
        worksheet_name[:31]
    )

    title_format = workbook.add_format({
        "bold": True,
        "font_size": 15,
    })

    header_format = workbook.add_format({
        "bold": True,
        "font_color": "#FFFFFF",
        "bg_color": "#1F4E78",
        "border": 1,
        "text_wrap": True,
        "valign": "top",
    })

    date_format = workbook.add_format({
        "num_format": "yyyy-mm-dd",
    })

    datetime_format = workbook.add_format({
        "num_format": "yyyy-mm-dd hh:mm:ss",
    })

    integer_format = workbook.add_format({
        "num_format": "0",
    })

    decimal_format = workbook.add_format({
        "num_format": "0.0000",
    })

    percent_format = workbook.add_format({
        "num_format": "0.00%",
    })

    # --------------------------------------------------------
    # Title and metadata
    # --------------------------------------------------------

    title_row = 0
    header_row = 2

    worksheet.write(
        title_row,
        0,
        (
            title
            or "Global Automotive Research Dataset"
        ),
        title_format,
    )

    worksheet.write(
        1,
        0,
        "Exported at UTC",
    )

    worksheet.write(
        1,
        1,
        datetime.now(
            timezone.utc
        )
        .replace(
            tzinfo=None
        ),
        datetime_format,
    )

    # --------------------------------------------------------
    # Headers
    # --------------------------------------------------------

    columns = list(
        dataframe.columns
    )

    for column_index, column_name in enumerate(
        columns
    ):
        worksheet.write(
            header_row,
            column_index,
            str(column_name),
            header_format,
        )

    # --------------------------------------------------------
    # Data rows
    # --------------------------------------------------------

    for excel_row, values in enumerate(
        dataframe.itertuples(
            index=False,
            name=None,
        ),
        start=header_row + 1,
    ):
        for column_index, value in enumerate(
            values
        ):
            safe_value = excel_safe_value(
                value
            )

            if safe_value is None:
                continue

            if isinstance(
                safe_value,
                datetime,
            ):
                worksheet.write_datetime(
                    excel_row,
                    column_index,
                    safe_value,
                    datetime_format,
                )

            elif isinstance(
                safe_value,
                date,
            ):
                worksheet.write_datetime(
                    excel_row,
                    column_index,
                    datetime.combine(
                        safe_value,
                        datetime.min.time(),
                    ),
                    date_format,
                )

            elif isinstance(
                safe_value,
                bool,
            ):
                worksheet.write_boolean(
                    excel_row,
                    column_index,
                    safe_value,
                )

            elif isinstance(
                safe_value,
                int,
            ):
                worksheet.write_number(
                    excel_row,
                    column_index,
                    safe_value,
                    integer_format,
                )

            elif isinstance(
                safe_value,
                float,
            ):
                worksheet.write_number(
                    excel_row,
                    column_index,
                    safe_value,
                    decimal_format,
                )

            else:
                worksheet.write(
                    excel_row,
                    column_index,
                    safe_value,
                )

    # --------------------------------------------------------
    # Worksheet presentation
    # --------------------------------------------------------

    final_row = (
        header_row
        + len(dataframe)
    )

    final_column = max(
        0,
        len(columns) - 1,
    )

    worksheet.freeze_panes(
        header_row + 1,
        4,
    )

    worksheet.autofilter(
        header_row,
        0,
        final_row,
        final_column,
    )

    worksheet.set_row(
        header_row,
        32,
    )

    # Identifier columns.
    worksheet.set_column(
        0,
        min(
            7,
            final_column,
        ),
        21,
    )

    # General data columns.
    if final_column >= 8:
        worksheet.set_column(
            8,
            final_column,
            16,
        )

    workbook.close()

    return {
        "output_path":
            str(output_path),

        "rows_written":
            len(dataframe),

        "columns_written":
            len(columns),

        "file_size_bytes":
            output_path.stat().st_size,

        "file_size_mb":
            output_path.stat().st_size
            / (1024 ** 2),

        "status":
            "PASSED",
    }


# ------------------------------------------------------------
# 5. Export each regional production dataset
# ------------------------------------------------------------

regional_excel_export_rows = []


if EXPORT_GLOBAL_DATASET_TO_EXCEL:
    print("=" * 92)
    print("EXPORTING GLOBAL DATASET BY REGION")
    print("=" * 92)

    print(
        "Export folder:",
        GLOBAL_EXCEL_EXPORT_DIR,
    )

    selected_export_df = (
        global_fundamentals_selected_df
    )

    if (
        "source_region"
        not in selected_export_df.columns
    ):
        raise RuntimeError(
            "global_fundamentals_selected_df does not "
            "contain source_region."
        )

    source_regions = (
        selected_export_df[
            "source_region"
        ]
        .dropna()
        .astype("string")
        .unique()
        .tolist()
    )

    for source_region in source_regions:
        region_mask = (
            selected_export_df[
                "source_region"
            ]
            .astype("string")
            .eq(
                source_region
            )
            .fillna(False)
        )

        region_df = (
            selected_export_df.loc[
                region_mask
            ]
        )

        output_filename = (
            REGION_FILE_NAMES.get(
                source_region,
                (
                    "Global_Automotive_Dataset_"
                    + re.sub(
                        r"[^A-Za-z0-9]+",
                        "_",
                        str(source_region),
                    ).strip("_")
                    + ".xlsx"
                ),
            )
        )

        output_path = (
            GLOBAL_EXCEL_EXPORT_DIR
            / output_filename
        )

        print(
            f"\nExporting {source_region}: "
            f"{len(region_df):,} rows..."
        )

        export_result = (
            write_dataframe_streaming(
                dataframe=region_df,
                output_path=output_path,
                worksheet_name="Global_Dataset",
                title=(
                    "Global Automotive Research Dataset — "
                    + str(source_region)
                ),
            )
        )

        export_result[
            "source_region"
        ] = source_region

        regional_excel_export_rows.append(
            export_result
        )

        print(
            "Completed:",
            output_path.name,
            f"({export_result['file_size_mb']:,.1f} MB)",
        )

        del region_df
        gc.collect()


regional_excel_export_report_df = (
    pd.DataFrame(
        regional_excel_export_rows
    )
)


display(
    regional_excel_export_report_df[
        [
            "source_region",
            "rows_written",
            "columns_written",
            "file_size_mb",
            "status",
            "output_path",
        ]
    ]
)


# ------------------------------------------------------------
# 6. Create compact summary workbook
# ------------------------------------------------------------

if SUMMARY_WORKBOOK_PATH.exists():
    SUMMARY_WORKBOOK_PATH.unlink()


summary_metadata_df = pd.DataFrame({
    "field": [
        "exported_at_utc",
        "canonical_dataset",
        "selected_fact_rows",
        "selected_fact_columns",
        "combined_fact_rows",
        "alternative_fact_rows",
        "monthly_panel_rows",
        "latent_factor_rows",
        "quantconnect_long_rows",
        "regional_workbooks",
        "export_directory",
        "canonical_storage_note",
    ],
    "value": [
        datetime.now(
            timezone.utc
        ).isoformat(),

        "global_fundamentals_selected_df",

        len(
            global_fundamentals_selected_df
        ),

        len(
            global_fundamentals_selected_df.columns
        ),

        len(
            global_fundamentals_combined_df
        ),

        len(
            global_fundamentals_alternatives_df
        ),

        len(
            global_monthly_fundamental_panel_df
        ),

        len(
            latent_factor_model_matrix_df
        ),

        len(
            quantconnect_fundamental_long_df
        ),

        len(
            regional_excel_export_report_df
        ),

        str(
            GLOBAL_EXCEL_EXPORT_DIR
        ),

        (
            "Parquet remains the canonical machine-readable "
            "format. Regional Excel files are inspection and "
            "sharing copies."
        ),
    ],
})


with pd.ExcelWriter(
    SUMMARY_WORKBOOK_PATH,
    engine="xlsxwriter",
    engine_kwargs={
        "options": {
            "strings_to_urls": False,
            "nan_inf_to_errors": False,
        }
    },
    datetime_format="yyyy-mm-dd hh:mm:ss",
    date_format="yyyy-mm-dd",
) as writer:

    summary_metadata_df.to_excel(
        writer,
        sheet_name="README",
        index=False,
    )

    regional_excel_export_report_df.to_excel(
        writer,
        sheet_name="Regional_Files",
        index=False,
    )

    regional_coverage_df.to_excel(
        writer,
        sheet_name="Regional_Coverage",
        index=False,
    )

    block_9_feedback_by_region_df.to_excel(
        writer,
        sheet_name="Block9_Feedback",
        index=False,
    )

    block_10_final_pipeline_checks_df.to_excel(
        writer,
        sheet_name="Final_Checks",
        index=False,
    )

    global_feature_coverage_df.to_excel(
        writer,
        sheet_name="Feature_Coverage",
        index=False,
    )

    global_research_usability_report_df.to_excel(
        writer,
        sheet_name="Research_Usability",
        index=False,
    )

    model_export_validation_df.to_excel(
        writer,
        sheet_name="Model_Validation",
        index=False,
    )

    block_10_validation_report_df.to_excel(
        writer,
        sheet_name="Persistence",
        index=False,
    )

    workbook = writer.book

    header_format = workbook.add_format({
        "bold": True,
        "font_color": "#FFFFFF",
        "bg_color": "#1F4E78",
        "border": 1,
        "text_wrap": True,
    })

    for worksheet_name, worksheet in (
        writer.sheets.items()
    ):
        worksheet.freeze_panes(
            1,
            0,
        )

        worksheet.set_row(
            0,
            28,
            header_format,
        )

        worksheet.set_column(
            0,
            max(
                1,
                worksheet.dim_colmax,
            ),
            22,
        )


summary_file_size_bytes = (
    SUMMARY_WORKBOOK_PATH.stat().st_size
)


block_10_excel_export_summary_df = pd.DataFrame([
    {
        "export_directory":
            str(
                GLOBAL_EXCEL_EXPORT_DIR
            ),

        "summary_workbook":
            str(
                SUMMARY_WORKBOOK_PATH
            ),

        "regional_workbooks":
            len(
                regional_excel_export_report_df
            ),

        "total_rows_exported":
            int(
                regional_excel_export_report_df[
                    "rows_written"
                ].sum()
            ),

        "expected_selected_rows":
            len(
                global_fundamentals_selected_df
            ),

        "row_reconciliation_difference":
            (
                int(
                    regional_excel_export_report_df[
                        "rows_written"
                    ].sum()
                )
                - len(
                    global_fundamentals_selected_df
                )
            ),

        "summary_file_size_mb":
            summary_file_size_bytes
            / (1024 ** 2),

        "all_regional_exports_passed":
            regional_excel_export_report_df[
                "status"
            ]
            .astype("string")
            .eq("PASSED")
            .all(),

        "status":
            (
                "PASSED"
                if (
                    regional_excel_export_report_df[
                        "status"
                    ]
                    .astype("string")
                    .eq("PASSED")
                    .all()
                    and int(
                        regional_excel_export_report_df[
                            "rows_written"
                        ].sum()
                    )
                    == len(
                        global_fundamentals_selected_df
                    )
                )
                else "FAILED"
            ),
    }
])


print("\n" + "=" * 92)
print("REGIONAL EXCEL EXPORT COMPLETE")
print("=" * 92)

display(
    block_10_excel_export_summary_df
)


if (
    block_10_excel_export_summary_df[
        "status"
    ].iloc[0]
    != "PASSED"
):
    raise RuntimeError(
        "The regional Excel export did not reconcile "
        "to global_fundamentals_selected_df."
    )


print(
    "Export folder:",
    GLOBAL_EXCEL_EXPORT_DIR,
)

print(
    "Summary workbook:",
    SUMMARY_WORKBOOK_PATH,
)

In [ ]:
# ============================================================
# 23. GLOBAL SCHEMA COMPLETENESS AUDIT
# ============================================================

print("=" * 80)
print("GLOBAL SCHEMA COMPLETENESS AUDIT")
print("=" * 80)

REQUIRED_GLOBAL_FIELDS = [

    # --------------------------------------------------------
    # Security identifiers
    # --------------------------------------------------------

    "issuer_id",
    "security_id",
    "issuer_name",
    "ticker",

    # --------------------------------------------------------
    # Filing identifiers
    # --------------------------------------------------------

    "document_id",
    "filing_id",

    # --------------------------------------------------------
    # Accounting metadata
    # --------------------------------------------------------

    "statement_type",
    "period_type",
    "fiscal_year",
    "fiscal_period",
    "period_start",
    "period_end",

    # --------------------------------------------------------
    # Standardisation
    # --------------------------------------------------------

    "standard_concept",
    "reported_value",
    "reported_currency",

    # --------------------------------------------------------
    # Expected interpretation
    # --------------------------------------------------------

    "expected_period_type",
    "expected_unit_family",

    # --------------------------------------------------------
    # Actual interpretation
    # --------------------------------------------------------

    "unit_family",

    # --------------------------------------------------------
    # Lineage
    # --------------------------------------------------------

    "source_region",
    "source_table",
    "source_row_number",
    "global_source_observation_id",

    # --------------------------------------------------------
    # Timing
    # --------------------------------------------------------

    "available_datetime",

]

schema_rows = []

regions = sorted(
    global_fundamentals_combined_df["source_region"]
    .dropna()
    .unique()
)

for region in regions:

    regional = global_fundamentals_combined_df.loc[
        global_fundamentals_combined_df["source_region"] == region
    ]

    total_rows = len(regional)

    for column in REQUIRED_GLOBAL_FIELDS:

        if column not in regional.columns:

            non_null = 0
            pct = 0.0

        else:

            non_null = regional[column].notna().sum()

            pct = (
                100 * non_null / total_rows
                if total_rows > 0
                else 0
            )

        schema_rows.append({

            "region": region,
            "column": column,
            "non_null_rows": non_null,
            "total_rows": total_rows,
            "coverage_percent": round(pct,2),

        })

global_schema_completeness_df = (
    pd.DataFrame(schema_rows)
)

display(
    global_schema_completeness_df
)

print()
print("=" * 80)
print("SCHEMA COMPLETENESS MATRIX")
print("=" * 80)

schema_matrix_df = (
    global_schema_completeness_df
    .pivot(
        index="column",
        columns="region",
        values="coverage_percent",
    )
    .fillna(0)
)

display(schema_matrix_df)

GLOBAL SCHEMA COMPLETENESS AUDIT


,region,column,non_null_rows,total_rows,coverage_percent
0,AUSTRALIA,issuer_id,659,659,100.00
1,AUSTRALIA,security_id,659,659,100.00
2,AUSTRALIA,issuer_name,659,659,100.00
3,AUSTRALIA,ticker,659,659,100.00
4,AUSTRALIA,document_id,659,659,100.00
5,AUSTRALIA,filing_id,659,659,100.00
6,AUSTRALIA,statement_type,433,659,65.71
7,AUSTRALIA,period_type,659,659,100.00
8,AUSTRALIA,fiscal_year,0,659,0.00
9,AUSTRALIA,fiscal_period,0,659,0.00



SCHEMA COMPLETENESS MATRIX


region,AUSTRALIA,EUROPE,HONG_KONG,JAPAN,KOREA,MAINLAND_CHINA,USA
column,,,,,,,
available_datetime,98.48,100.00,100.0,100.00,48.76,100.0,100.00
document_id,100.00,100.00,100.0,100.00,100.00,100.0,100.00
expected_period_type,65.71,100.00,100.0,100.00,100.00,100.0,100.00
expected_unit_family,65.71,100.00,100.0,100.00,100.00,100.0,100.00
filing_id,100.00,100.00,100.0,100.00,100.00,100.0,100.00
fiscal_period,0.00,0.00,0.0,0.00,0.00,0.0,99.86
fiscal_year,0.00,0.00,0.0,0.00,0.00,0.0,99.86
global_source_observation_id,100.00,100.00,100.0,100.00,100.00,100.0,100.00
issuer_id,100.00,100.00,100.0,100.00,100.00,100.0,94.99


## Architectural notes

### Block 9 integration

Block 10 loads the deduplicated Block 9 production outcomes where available. Only rows marked as accepted through `signature_decision_applied` are applied. Original source concepts and values remain available alongside the effective fields and Block 9 decision metadata.

### Point-in-time integrity

The Block 9 overlay is applied before economic fact keys, preferred-source selection and monthly point-in-time panels are constructed. This ensures accepted concept corrections flow consistently into duplicate resolution, feature engineering and research exports without introducing look-ahead bias.

### Latent-factor and QuantConnect use

`latent_factor_model_matrix_df` remains the issuer-month accounting-characteristic matrix. `quantconnect_fundamental_long_df` carries effective concepts and values for LEAN custom data or Object Store ingestion. Permanent issuer and security identifiers remain separate from tradable symbols.
